# DKD Risk Prediction in Type 2 Diabetes
## 01. Data Curation

This notebook curates the landmark-independent data used to construct diabetic kidney disease (DKD) prediction cohorts in the All of Us Research Program.

The workflow:

1. Curates participant demographics and EHR observation windows.
2. Defines the first evidence of type 2 diabetes (T2D).
3. Curates laboratory and diagnosis-based kidney disease evidence.
4. Identifies persistent albuminuria and persistent reduced eGFR.
5. Identifies dialysis and kidney transplantation.
6. Builds reusable longitudinal UACR and eGFR measurement tables.
7. Saves validated inputs for cohort construction.

Case/control assignment and prediction landmarks are defined separately in `02_cohort_construction.ipynb`.


# Setup


In [ ]:
import json
import os
import re
import subprocess
import warnings

import gcsfs
import numpy as np
import pandas as pd
from google.cloud import bigquery

warnings.filterwarnings(
    "ignore",
    message="Unable to represent RANGE schema as struct using pandas ArrowDtype",
)

# ============================================================
# Migrated All of Us workspace
# ============================================================

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

CURATION_DIR = f"{bucket}/aou_dkd/dkd_data_curation_20260821"

# Existing migrated project outputs used only when an upstream
# artifact is required as an input.
SOURCE_PHENOTYPE_DIR = f"{bucket}/aou_dkd/dkd_phenotypes_v3"
SOURCE_PHENOTYPE_DIR_LEGACY = f"{bucket}/aou_dkd/dkd_phenotypes"

fs = gcsfs.GCSFileSystem()


# ============================================================
# Resolve the All of Us CDR BigQuery resource
# ============================================================

def resolve_aou_cdr():
    """
    Resolve the All of Us CDR attached to the current Verily workspace.

    Verily Workbench does not automatically provide the legacy
    WORKSPACE_CDR environment variable. Instead, BigQuery datasets
    are workspace resources and are resolved with the Workbench CLI.

    The DKD project uses Controlled Tier data, so this function looks
    for a Controlled Tier CDR with an identifier such as C2024Q3R8.
    """

    result = subprocess.run(
        ["wb", "resource", "list", "--type=BQ_DATASET", "--format=JSON"],
        capture_output=True,
        text=True,
        check=True,
    )

    resources = json.loads(result.stdout)

    cdr_resources = []

    for resource in resources:
        dataset_id = str(resource.get("datasetId", ""))

        # All of Us Controlled Tier CDR identifiers follow this pattern.
        if re.fullmatch(r"C\d{4}Q\d+R\d+", dataset_id):
            project_id = resource.get("projectId")

            if project_id:
                cdr_resources.append(
                    {
                        "resource_id": resource.get("id"),
                        "project_id": project_id,
                        "dataset_id": dataset_id,
                        "path": f"{project_id}.{dataset_id}",
                    }
                )

    if len(cdr_resources) == 0:
        raise RuntimeError(
            "No All of Us Controlled Tier CDR BigQuery resource was found "
            "in this workspace. Add the appropriate Controlled Tier CDR "
            "resource from the All of Us data collection before running "
            "this notebook."
        )

    if len(cdr_resources) > 1:
        candidates = "\n".join(
            f"  {x['resource_id']}: {x['path']}"
            for x in cdr_resources
        )

        raise RuntimeError(
            "Multiple All of Us Controlled Tier CDR resources were found.\n"
            "For reproducibility, the notebook will not choose one automatically.\n"
            "Available CDRs:\n"
            f"{candidates}\n\n"
            "Set CDR_DATASET explicitly to the intended project.dataset."
        )

    return cdr_resources[0]["path"]


CDR_DATASET = resolve_aou_cdr()

# Use the resolved CDR as the default dataset for BigQuery jobs.
job_query_config = bigquery.QueryJobConfig(
    default_dataset=CDR_DATASET
)

bq_client = bigquery.Client(
    default_query_job_config=job_query_config
)

print("Workspace:", bucket)
print("Curated data directory:", CURATION_DIR)
print("All of Us CDR:", CDR_DATASET)

## Data I/O

All curated outputs are written directly to Google Cloud Storage. CSV is used for compact participant-level tables, while Parquet is used for large longitudinal measurement tables.


In [ ]:
def save_gcs_csv(df, path, **kwargs):
    """Save a DataFrame as CSV to Google Cloud Storage."""
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False, **kwargs)

    print(
        f"Saved CSV: {path}\n"
        f"  rows: {len(df):,}\n"
        f"  columns: {df.shape[1]:,}"
    )


def read_gcs_csv(path, **kwargs):
    """Read a CSV directly from Google Cloud Storage."""
    with fs.open(path, "rb") as f:
        return pd.read_csv(f, **kwargs)


def save_gcs_parquet(df, path, **kwargs):
    """Save a DataFrame as Parquet to Google Cloud Storage."""
    with fs.open(path, "wb") as f:
        df.to_parquet(f, index=False, **kwargs)

    print(
        f"Saved Parquet: {path}\n"
        f"  rows: {len(df):,}\n"
        f"  columns: {df.shape[1]:,}"
    )


def read_gcs_parquet(path, **kwargs):
    """Read a Parquet dataset directly from Google Cloud Storage."""
    with fs.open(path, "rb") as f:
        return pd.read_parquet(f, **kwargs)


# Compatibility wrappers used by retained extraction code below.
# Every call writes into the new data-curation directory.
def save_phenotypes(df, bucket, filename, **kwargs):
    path = f"{CURATION_DIR}/{filename}"
    save_gcs_csv(df, path)
    return path


def save_phenotypes_fast(df, bucket, filename, **kwargs):
    base = filename
    for ext in (".parquet", ".csv", ".csv.gz", ".feather"):
        if base.endswith(ext):
            base = base[:-len(ext)]
            break

    path = f"{CURATION_DIR}/{base}.parquet"
    save_gcs_parquet(df, path)
    return path


def read_first_existing_gcs_csv(paths, **kwargs):
    """Read the first existing GCS CSV from an ordered list of candidate paths."""
    for path in paths:
        if fs.exists(path):
            print(f"Loading: {path}")
            return read_gcs_csv(path, **kwargs)

    raise FileNotFoundError(
        "None of the candidate GCS paths exist:\n" + "\n".join(paths)
    )


## Common Cleaning Functions

Participant identifiers and dates are normalized consistently before phenotype assembly and temporal comparisons.


In [ ]:
BAD_DATE_TOKENS = {
    "",
    "null",
    "none",
    "nan",
    "na",
    "n/a",
    "nat",
    "0000-00-00",
    "0000-00-00 00:00:00",
}


def parse_datetime(series):
    """Parse a date-like series as UTC datetime."""
    s = pd.Series(series)

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s, errors="coerce", utc=True)

    s = s.astype("string").str.strip()
    s = s.where(~s.str.lower().isin(BAD_DATE_TOKENS), pd.NA)

    parsed = pd.to_datetime(s, errors="coerce", utc=True)

    # Recover an embedded YYYY-MM-DD when the complete value cannot be parsed.
    missing = parsed.isna() & s.notna()
    if missing.any():
        ymd = s.loc[missing].str.extract(
            r"(\d{4}-\d{2}-\d{2})",
            expand=False,
        )
        parsed.loc[missing] = pd.to_datetime(
            ymd,
            errors="coerce",
            utc=True,
        )

    return parsed


def normalize_person_id(df, label="data"):
    """Standardize person_id as nullable Int64 and remove invalid IDs."""
    if "person_id" not in df.columns:
        raise KeyError(f"{label}: missing person_id")

    out = df.copy()
    out["person_id"] = pd.to_numeric(
        out["person_id"],
        errors="coerce",
    ).astype("Int64")

    n_invalid = int(out["person_id"].isna().sum())
    if n_invalid:
        print(f"{label}: removing {n_invalid:,} rows with invalid person_id")
        out = out.dropna(subset=["person_id"]).copy()

    return out


# 1. Participant Data


## Demographics

Participant demographics are extracted from the OMOP `person` table. Date of birth and sex at birth are also used to calculate creatinine-based eGFR.


In [ ]:
dataset_all_person_sql = f"""
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name AS gender,
        person.birth_datetime AS date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name AS race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name AS ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name AS sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name AS self_reported_category
    FROM
        `{CDR_DATASET}.person` person
    LEFT JOIN
        `{CDR_DATASET}.concept` p_gender_concept
        ON person.gender_concept_id = p_gender_concept.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` p_race_concept
        ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` p_ethnicity_concept
        ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` p_sex_at_birth_concept
        ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` p_self_reported_category_concept
        ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id
"""
dataset_all_person_df = pd.read_gbq(
    dataset_all_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)


In [ ]:
dataset_all_person_df = normalize_person_id(
    dataset_all_person_df,
    label="demographics",
)

print(f"Participants: {dataset_all_person_df['person_id'].nunique():,}")
print(dataset_all_person_df["sex_at_birth"].value_counts(dropna=False))

save_gcs_parquet(
    dataset_all_person_df,
    f"{CURATION_DIR}/participant_demographics.parquet",
)


## EHR Observation Window

The EHR observation period is defined using the earliest and latest dated clinical records across visits, conditions, measurements, drug exposures, and procedures.


In [ ]:
sql_ehr_window = f"""
WITH all_dates AS (

  SELECT person_id, visit_start_datetime AS dt
  FROM `{CDR_DATASET}.visit_occurrence`
  WHERE visit_start_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, visit_end_datetime AS dt
  FROM `{CDR_DATASET}.visit_occurrence`
  WHERE visit_end_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, condition_start_datetime AS dt
  FROM `{CDR_DATASET}.condition_occurrence`
  WHERE condition_start_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, measurement_datetime AS dt
  FROM `{CDR_DATASET}.measurement`
  WHERE measurement_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, drug_exposure_start_datetime AS dt
  FROM `{CDR_DATASET}.drug_exposure`
  WHERE drug_exposure_start_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, drug_exposure_end_datetime AS dt
  FROM `{CDR_DATASET}.drug_exposure`
  WHERE drug_exposure_end_datetime IS NOT NULL

  UNION ALL
  SELECT person_id, procedure_datetime AS dt
  FROM `{CDR_DATASET}.procedure_occurrence`
  WHERE procedure_datetime IS NOT NULL
)

SELECT
  person_id,
  MIN(dt) AS ehr_start_date,
  MAX(dt) AS ehr_end_date
FROM all_dates
GROUP BY person_id
"""

print("Running EHR observation-window query...")
ehr_dates = bq_client.query(sql_ehr_window).result().to_dataframe()

ehr_dates = normalize_person_id(ehr_dates, label="EHR dates")
ehr_dates["ehr_start_date"] = parse_datetime(ehr_dates["ehr_start_date"])
ehr_dates["ehr_end_date"] = parse_datetime(ehr_dates["ehr_end_date"])

assert ehr_dates["person_id"].is_unique
assert (
    ehr_dates["ehr_end_date"] >= ehr_dates["ehr_start_date"]
).all()

duration_years = (
    ehr_dates["ehr_end_date"] - ehr_dates["ehr_start_date"]
).dt.days / 365.25

print(f"Participants with EHR data: {len(ehr_dates):,}")
print("\nEHR duration in years:")
print(duration_years.describe())

save_phenotypes(
    ehr_dates,
    bucket,
    "derived_ehr_dates.csv",
)


# 2. Survey Data

Core All of Us surveys are curated into participant-level tables for later feature construction.


In [ ]:
def transform_single_answer_survey(df, date_col_name):
    """Convert a single-answer survey to one row per participant."""
    x = df.copy()

    x["Answer"] = x["answer"].astype("string").apply(
        lambda value: value.split(":", 1)[1].strip()
        if pd.notna(value) and ":" in value
        else value
    )

    x["Column"] = (
        x["question"]
        .astype("string")
        .str.replace(":", "", regex=False)
        .str.replace("'", "", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    wide = (
        x.groupby(["person_id", "Column"])["Answer"]
        .first()
        .unstack()
        .reset_index()
        .rename_axis(None, axis=1)
    )

    dates = (
        x.assign(survey_datetime=parse_datetime(x["survey_datetime"]))
        .groupby("person_id", as_index=False)["survey_datetime"]
        .min()
        .rename(columns={"survey_datetime": date_col_name})
    )

    return wide.merge(dates, on="person_id", how="left")


def transform_multiselect_survey(df, date_col_name):
    """Convert a multi-select survey to one row per participant."""
    x = df.copy()

    x["Answer"] = x["answer"].astype("string").apply(
        lambda value: value.split("-", 1)[1].strip()
        if pd.notna(value) and "-" in value
        else value
    )

    x["Column"] = (
        x["question"]
        .astype("string")
        .str.replace(r"\([^)]*\)", "", regex=True)
        .str.replace(" Select all that apply.", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(":", "", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace("'", "", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    wide = (
        x.groupby(["person_id", "Column"])["Answer"]
        .agg(list)
        .unstack()
        .reset_index()
        .rename_axis(None, axis=1)
    )

    dates = (
        x.assign(survey_datetime=parse_datetime(x["survey_datetime"]))
        .groupby("person_id", as_index=False)["survey_datetime"]
        .min()
        .rename(columns={"survey_datetime": date_col_name})
    )

    return wide.merge(dates, on="person_id", how="left")


In [ ]:
def extract_survey(survey_name):
    sql = f"""
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name
    FROM `{CDR_DATASET}.ds_survey` AS answer
    WHERE answer.survey = @survey_name
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "survey_name",
                "STRING",
                survey_name,
            )
        ]
    )

    return bq_client.query(
        sql,
        job_config=job_config,
    ).to_dataframe()


### The Basics


In [ ]:
basics_survey_df = extract_survey("The Basics")
transformed_basics_survey_df = transform_single_answer_survey(
    basics_survey_df,
    date_col_name="basics_survey_date",
)

print(
    f"The Basics: {transformed_basics_survey_df['person_id'].nunique():,} participants"
)

save_phenotypes(
    transformed_basics_survey_df,
    bucket,
    "aou_basic_surveys.csv",
)


### Social Determinants of Health


In [ ]:
social_hlth_survey_df = extract_survey("Social Determinants of Health")
transformed_social_hlth_survey_df = transform_single_answer_survey(
    social_hlth_survey_df,
    date_col_name="social_health_survey_date",
)

print(
    f"Social Determinants of Health: "
    f"{transformed_social_hlth_survey_df['person_id'].nunique():,} participants"
)

save_phenotypes(
    transformed_social_hlth_survey_df,
    bucket,
    "aou_social_health_surveys.csv",
)


### Lifestyle


In [ ]:
lifestyle_survey_df = extract_survey("Lifestyle")
transformed_lifestyle_survey_df = transform_single_answer_survey(
    lifestyle_survey_df,
    date_col_name="lifestyle_survey_date",
)

print(
    f"Lifestyle: {transformed_lifestyle_survey_df['person_id'].nunique():,} participants"
)

save_phenotypes(
    transformed_lifestyle_survey_df,
    bucket,
    "aou_lifestyle_surveys.csv",
)


### Personal and Family Health History


In [ ]:
personal_fam_his_survey_df = extract_survey(
    "Personal and Family Health History"
)

transformed_personal_fam_his_survey_df = transform_multiselect_survey(
    personal_fam_his_survey_df,
    date_col_name="personal_family_history_survey_date",
)

print(
    f"Personal and Family Health History: "
    f"{transformed_personal_fam_his_survey_df['person_id'].nunique():,} participants"
)

save_phenotypes(
    transformed_personal_fam_his_survey_df,
    bucket,
    "aou_personal_family_history_surveys.csv",
)


In [ ]:
# Self-reported T2D is retained as a supportive survey variable.
fh_t2d_col = (
    "Including_yourself_who_in_your_family_has_had_type_2_diabetes?"
)

if fh_t2d_col in transformed_personal_fam_his_survey_df.columns:
    self_t2d = transformed_personal_fam_his_survey_df[
        ["person_id", fh_t2d_col]
    ].copy()

    self_t2d["Self_T2D"] = (
        self_t2d[fh_t2d_col]
        .fillna("")
        .astype(str)
        .str.contains("Self", regex=False)
        .astype(int)
    )

    save_phenotypes(
        self_t2d,
        bucket,
        "aou_survey_self_reported_t2d.csv",
    )
else:
    print(
        "Self-reported T2D question not found in the transformed survey; "
        "continuing without the supportive self-report flag."
    )


# 3. T2D Ascertainment

The T2D index date is defined as the earliest qualifying evidence from diagnosis codes, HbA1c, fasting blood glucose, or random blood glucose.


## Measurement Extraction Functions

Laboratory concepts are discovered from the OMOP concept table and then restricted to curated concept and unit allowlists before extraction.


In [ ]:
from google.cloud import bigquery
import pandas as pd
import os

def discover_lab_concepts_by_name(
    pattern: str,
    workspace_cdr: str | None = None,
    limit: int = 200,
    use_regex: bool = True,
    include_source_concepts: bool = False,
) -> pd.DataFrame:
    """
    Discover which measurement_concept_id(s) in your OMOP `measurement` table
    correspond to a lab of interest by searching the STANDARD concept_name.

    Intended use:
      1) run this once to identify the dominant concept_ids used in your dataset
      2) curate an allowlist of concept_ids
      3) extract measurements by concept_id allowlist (stable + reproducible)

    Parameters
    ----------
    pattern : str
        Search pattern for concept_name. If use_regex=True, treated as a regex
        applied to LOWER(concept_name). If use_regex=False, used as substring.
        Examples:
          - r'\\bhba1c\\b|hemoglobin a1c|\\ba1c\\b'
          - 'hba1c'
    workspace_cdr : str | None
        Dataset path. Defaults to CDR_DATASET.
    limit : int
        Max number of concept rows to return.
    use_regex : bool
        If True, uses REGEXP_CONTAINS. If False, uses LIKE with %pattern%.
    include_source_concepts : bool
        If True, also searches `measurement_source_value` / source concept fields
        to help diagnose mapping gaps. This is noisier but useful for auditing.

    Returns
    -------
    pd.DataFrame with columns:
      - measurement_concept_id
      - concept_name
      - vocabulary_id
      - concept_class_id
      - standard_concept
      - n_rows
      - n_people
    (plus optional source fields summary if include_source_concepts=True)
    """
    client = bigquery.Client()
    workspace = workspace_cdr or CDR_DATASET

    if use_regex:
        concept_filter = "REGEXP_CONTAINS(LOWER(c.concept_name), @pattern)"
    else:
        concept_filter = "LOWER(c.concept_name) LIKE CONCAT('%', LOWER(@pattern), '%')"

    # Optional: source-based search (auditing mapping issues)
    source_union = ""
    if include_source_concepts:
        # This can pull a lot of noise; use only for debugging/coverage checks.
        source_union = f"""
        UNION ALL
        SELECT
          COALESCE(m.measurement_source_concept_id, -1) AS measurement_concept_id,
          COALESCE(c2.concept_name, m.measurement_source_value) AS concept_name,
          COALESCE(c2.vocabulary_id, 'SOURCE') AS vocabulary_id,
          COALESCE(c2.concept_class_id, 'SOURCE') AS concept_class_id,
          COALESCE(c2.standard_concept, 'S') AS standard_concept,
          COUNT(*) AS n_rows,
          COUNT(DISTINCT m.person_id) AS n_people
        FROM `{workspace}.measurement` m
        LEFT JOIN `{workspace}.concept` c2
          ON m.measurement_source_concept_id = c2.concept_id
        WHERE m.measurement_source_value IS NOT NULL
          AND (
            REGEXP_CONTAINS(LOWER(m.measurement_source_value), @pattern)
          )
        GROUP BY 1,2,3,4,5
        """

    sql = f"""
    WITH standard_hits AS (
      SELECT
        m.measurement_concept_id,
        c.concept_name,
        c.vocabulary_id,
        c.concept_class_id,
        c.standard_concept,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT m.person_id) AS n_people
      FROM `{workspace}.measurement` m
      JOIN `{workspace}.concept` c
        ON m.measurement_concept_id = c.concept_id
      WHERE {concept_filter}
      GROUP BY 1,2,3,4,5
    )
    SELECT *
    FROM (
      SELECT * FROM standard_hits
      {source_union}
    )
    ORDER BY n_people DESC, n_rows DESC
    LIMIT @limit
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("pattern", "STRING", pattern),
            bigquery.ScalarQueryParameter("limit", "INT64", int(limit)),
        ]
    )

    df = client.query(sql, job_config=job_config).to_dataframe()

    # Slight cleanup: keep measurement_concept_id as Int64 (nullable)
    if "measurement_concept_id" in df.columns:
        df["measurement_concept_id"] = pd.to_numeric(df["measurement_concept_id"], errors="coerce").astype("Int64")

    return df


In [ ]:
from __future__ import annotations

from google.cloud import bigquery
import pandas as pd
import numpy as np
import os
from typing import Callable, Iterable, Sequence, Optional, Union, Any


def extract_numeric_measurements_from_discovered_concepts(
    concepts_df: pd.DataFrame,
    workspace_cdr: str | None = None,
    date_cutoffs: pd.DataFrame | None = None,
    concept_id_col: str = "measurement_concept_id",
    concept_name_col: str = "concept_name",
    vocabulary_col: str = "vocabulary_id",
    concept_class_col: str = "concept_class_id",
    # allowlist filters
    keep_vocab: tuple[str, ...] = ("LOINC",),
    keep_concept_classes: tuple[str, ...] = ("Lab Test",),
    drop_name_contains: tuple[str, ...] = (),
    allow_extra_rules: Sequence[dict[str, Any]] = (),
    # extraction options
    sentinel_values: tuple[float, ...] = (10000000.0,),
    require_units_containing: tuple[str, ...] | None = None,  # e.g. ("%",) or ("mg/dL",)
    allow_unit_nulls: bool = False,
    # plausibility bounds (applied after conversions)
    min_value: float | None = None,
    max_value: float | None = None,
    # unit conversions: list of rules applied in order
    # each rule: {"pattern": "mmol/mol", "new_unit": "% (converted)", "transform": callable}
    conversions: Sequence[dict[str, Any]] = (),
    # return options
    return_allowlist: bool = False,
    allowlist_sort_cols: tuple[str, ...] = ("n_people", "n_rows"),
) -> pd.DataFrame | tuple[pd.DataFrame, pd.DataFrame]:
    """
    General extractor for numeric OMOP measurement rows using a stable concept allowlist built
    from a discovery table (concepts_df).

    Required columns in concepts_df:
      - concept_id_col (default: measurement_concept_id)
      - concept_name_col (default: concept_name)
      - vocabulary_col (default: vocabulary_id)
      - concept_class_col (default: concept_class_id)

    Optional columns in concepts_df (for allowlist auditing / sorting):
      - n_people, n_rows (or whatever you pass in allowlist_sort_cols)

    Parameters
    ----------
    concepts_df:
        Discovery results dataframe (e.g., from a "discover_lab_concepts_by_name" query).
    workspace_cdr:
        BigQuery dataset. Defaults to CDR_DATASET.
    date_cutoffs:
        Optional per-person cutoff_date filter with columns: person_id, cutoff_date.
    keep_vocab / keep_concept_classes:
        Base allowlist filters.
    drop_name_contains:
        Substrings (case-insensitive) to exclude based on concept_name (e.g. ("goal",)).
    allow_extra_rules:
        Additional allowlist inclusions. Each dict can specify:
          {"vocabulary_id": "...", "concept_class_id": "..."} (plus optional name filters).
    sentinel_values:
        Values to drop (AoU often uses 10000000.0 as a sentinel).
    require_units_containing:
        If provided, keep rows whose unit_concept_name contains ANY of these substrings.
    allow_unit_nulls:
        If True, rows with null unit_concept_name are allowed even when unit filtering is on.
    min_value / max_value:
        Plausible numeric bounds after conversions.
    conversions:
        Sequence of conversion rules applied in order. Each rule dict must have:
          - "pattern": substring/regex to match in unit_concept_name (case-insensitive)
          - "transform": callable(x: pd.Series) -> pd.Series
        Optional:
          - "new_unit": string to replace unit_concept_name
        Example:
          {"pattern": "mmol/mol", "transform": lambda s: 0.09148*s + 2.152, "new_unit": "%"}
    return_allowlist:
        If True, return (df_measurements, allowlist_df).
    allowlist_sort_cols:
        Columns used to sort the allowlist for audit (if present).

    Returns
    -------
    df (and optionally allowlist_df)
    """
    client = bigquery.Client()
    workspace = workspace_cdr or CDR_DATASET

    dfc = concepts_df.copy()
    dfc["_name_lc"] = dfc[concept_name_col].astype(str).str.lower()

    # drop by name substrings
    for bad in drop_name_contains:
        dfc = dfc[~dfc["_name_lc"].str.contains(str(bad).lower(), na=False)]

    # base allowlist
    allow = dfc[
        dfc[vocabulary_col].isin(keep_vocab) &
        dfc[concept_class_col].isin(keep_concept_classes)
    ].copy()

    # extra allow rules (e.g., include SNOMED Procedure, or custom vocab/class combos)
    if allow_extra_rules:
        extra_frames = []
        for rule in allow_extra_rules:
            tmp = dfc.copy()
            if "vocabulary_id" in rule:
                tmp = tmp[tmp[vocabulary_col] == rule["vocabulary_id"]]
            if "concept_class_id" in rule:
                tmp = tmp[tmp[concept_class_col] == rule["concept_class_id"]]
            if "name_contains" in rule:
                # keep if contains any of these
                keep_any = False
                for s in rule["name_contains"]:
                    keep_any = keep_any | tmp["_name_lc"].str.contains(str(s).lower(), na=False)
                tmp = tmp[keep_any]
            if "name_not_contains" in rule:
                for s in rule["name_not_contains"]:
                    tmp = tmp[~tmp["_name_lc"].str.contains(str(s).lower(), na=False)]
            extra_frames.append(tmp)

        if extra_frames:
            allow = pd.concat([allow, *extra_frames], ignore_index=True)

    # unique concept_ids
    allow_ids = (
        pd.to_numeric(allow[concept_id_col], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if not allow_ids:
        raise ValueError("No concept_ids selected. Relax filters or inspect the discovery table.")

    # optional cutoff table
    cutoff_join_sql = ""
    where_cutoff = ""
    cutoff_table = None
    if date_cutoffs is not None:
        cut = date_cutoffs.copy()
        cut["cutoff_date"] = pd.to_datetime(cut["cutoff_date"], errors="coerce", utc=True)
        cut = cut.dropna(subset=["person_id", "cutoff_date"])
        cutoff_table = f"{workspace}._tmp_cutoffs_measurements"
        client.load_table_from_dataframe(cut, cutoff_table).result()
        cutoff_join_sql = f"JOIN `{cutoff_table}` cut ON m.person_id = cut.person_id"
        where_cutoff = "AND m.measurement_datetime <= cut.cutoff_date"

    sql = f"""
    SELECT
      m.person_id,
      m.measurement_concept_id,
      m.measurement_datetime,
      m.value_as_number,
      m.unit_concept_id,
      c.concept_name AS standard_concept_name,
      c.vocabulary_id AS standard_vocabulary,
      c.concept_class_id AS standard_concept_class,
      u.concept_name AS unit_concept_name
    FROM `{workspace}.measurement` m
    JOIN `{workspace}.concept` c
      ON m.measurement_concept_id = c.concept_id
    LEFT JOIN `{workspace}.concept` u
      ON m.unit_concept_id = u.concept_id
    {cutoff_join_sql}
    WHERE m.measurement_concept_id IN UNNEST(@concept_ids)
      AND m.measurement_datetime IS NOT NULL
      AND m.value_as_number IS NOT NULL
      {where_cutoff}
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("concept_ids", "INT64", allow_ids)]
    )
    df = client.query(sql, job_config=job_config).to_dataframe()

    if cutoff_table is not None:
        client.delete_table(cutoff_table, not_found_ok=True)

    # standardize types
    df["measurement_datetime"] = pd.to_datetime(df["measurement_datetime"], errors="coerce", utc=True)
    df["unit_concept_name"] = df["unit_concept_name"].astype("string")

    # drop sentinels
    if sentinel_values:
        df = df[~df["value_as_number"].isin(list(sentinel_values))]

    # conversions (apply before unit filtering/bounds)
    if conversions:
        for rule in conversions:
            pattern = rule["pattern"]
            transform = rule["transform"]
            new_unit = rule.get("new_unit", None)

            mask = df["unit_concept_name"].fillna("").str.contains(pattern, case=False, na=False)
            if mask.any():
                df.loc[mask, "value_as_number"] = transform(df.loc[mask, "value_as_number"])
                if new_unit is not None:
                    df.loc[mask, "unit_concept_name"] = new_unit

    # unit filtering
    if require_units_containing is not None:
        units = df["unit_concept_name"]
        if allow_unit_nulls:
            keep_mask = units.isna()
        else:
            keep_mask = pd.Series(False, index=df.index)

        match_any = pd.Series(False, index=df.index)
        for s in require_units_containing:
            match_any = match_any | units.fillna("").str.contains(str(s), case=False, na=False)

        df = df[keep_mask | match_any]

    # plausibility bounds
    if min_value is not None:
        df = df[df["value_as_number"] >= float(min_value)]
    if max_value is not None:
        df = df[df["value_as_number"] <= float(max_value)]

    # allowlist df for audit
    allowlist_df = allow.copy()
    for col in allowlist_sort_cols:
        if col not in allowlist_df.columns:
            # if missing, just skip sorting by it
            allowlist_sort_cols = tuple(c for c in allowlist_sort_cols if c in allowlist_df.columns)
            break
    if allowlist_sort_cols:
        allowlist_df = allowlist_df.sort_values(list(allowlist_sort_cols), ascending=False)
    allowlist_df = allowlist_df.reset_index(drop=True)

    return (df, allowlist_df) if return_allowlist else df


## T2D Diagnosis Codes

The established participant-level T2D diagnosis-code table is loaded from the migrated project storage and copied into the curated output directory.


In [ ]:
t2d_icd_earliest = read_first_existing_gcs_csv(
    [
        f"{SOURCE_PHENOTYPE_DIR}/t2d_codes_earliest_date.csv",
        f"{SOURCE_PHENOTYPE_DIR_LEGACY}/t2d_codes_earliest_date.csv",
    ]
)

t2d_icd_earliest = normalize_person_id(
    t2d_icd_earliest,
    label="T2D ICD",
)

if "condition_start_datetime" in t2d_icd_earliest.columns:
    t2d_icd_earliest["condition_start_datetime"] = parse_datetime(
        t2d_icd_earliest["condition_start_datetime"]
    )

print(
    f"T2D diagnosis-code participants: "
    f"{t2d_icd_earliest['person_id'].nunique():,}"
)

save_phenotypes(
    t2d_icd_earliest,
    bucket,
    "t2d_codes_earliest_date.csv",
)


## HbA1c


In [ ]:
a1c_concepts = discover_lab_concepts_by_name(
    pattern=r"\bhba1c\b|hemoglobin a1c|\ba1c\b",
    limit=200,
    use_regex=True,
    include_source_concepts=False  # flip to True if you suspect mapping gaps
)


In [ ]:
df_a1c, a1c_allow = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=a1c_concepts,
    date_cutoffs=None,
    keep_vocab=("LOINC", "SNOMED"),
    keep_concept_classes=("Lab Test", "Procedure"),
    drop_name_contains=("goal",),
    conversions=[
        {
            "pattern": r"(mmol/mol|millimole.*mole|ifcc)",
            "transform": lambda s: 0.09148 * s + 2.152,
            "new_unit": "percent (converted from IFCC)",
        }
    ],
    require_units_containing=("percent",),   # <-- key fix
    allow_unit_nulls=True,                   # <-- AoU-safe
    min_value=3.0,
    max_value=20.0,
    return_allowlist=True,
)


In [ ]:
# Save
save_phenotypes(df_a1c, bucket, "aou_a1c_labs.csv")


In [ ]:
# Filter only diabetic-range A1c values
diabetic_a1c = df_a1c[df_a1c['value_as_number'] >= 6.5].copy()

# Get the earliest (first) diabetic A1c per person
first_diabetic_a1c = (
    diabetic_a1c.sort_values(["person_id", "measurement_datetime"])
    .groupby("person_id")
    .first()
    .reset_index()
    .loc[:, ["person_id", "value_as_number", "measurement_datetime"]]
)

first_diabetic_a1c.rename(
    columns={"value_as_number": "first_diabetic_a1c",
             "measurement_datetime": "first_diabetic_a1c_date"},
    inplace=True
)

first_diabetic_a1c.head()


In [ ]:
# Save
save_phenotypes(first_diabetic_a1c, bucket, "person_with_diabetic_a1c.csv")


## Fasting Blood Glucose


In [ ]:
fbg_concepts = discover_lab_concepts_by_name(
    pattern=r"\bfasting\b.*\bglucose\b|\bglucose\b.*\bfasting\b|fasting plasma glucose",
    limit=500,
    use_regex=True,
    include_source_concepts=False
)

fbg_concepts.head(20)


In [ ]:
import pandas as pd

# -----------------------------
# Curate fasting blood glucose concept allowlist (exclude urine)
# -----------------------------
fbg_allowlist_df = (
    fbg_concepts
    .copy()
    .assign(concept_name_lc=lambda d: d["concept_name"].astype(str).str.lower())
)

# Drop urine-based concepts (presence / urine fasting) since they're not blood glucose values
fbg_allowlist_df = fbg_allowlist_df[
    ~fbg_allowlist_df["concept_name_lc"].str.contains(r"\burine\b", na=False)
].copy()

# -----------------------------
# Extract + clean fasting blood glucose measurements
# -----------------------------
df_fbg, fbg_allow_audit = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=fbg_allowlist_df,
    date_cutoffs=None,
    keep_vocab=("LOINC",),                 # anchor on LOINC Lab Test
    keep_concept_classes=("Lab Test",),
    drop_name_contains=("urine",),         # extra safety

    # include SNOMED fasting glucose procedures explicitly
    allow_extra_rules=[
        {"vocabulary_id": "SNOMED", "concept_class_id": "Procedure"}
    ],

    # convert mmol/L -> mg/dL (glucose)
    conversions=[
        {
            "pattern": r"(mmol|moles/volume)",
            "transform": lambda s: s * 18.0182,
            "new_unit": "mg/dL (converted from mmol/L)",
        }
    ],

    # keep mg/dL-like units; allow missing units but rely on plausible range QC
    require_units_containing=("mg/dl", "milligram per deciliter"),
    allow_unit_nulls=True,

    # plausible fasting glucose range (mg/dL)
    min_value=40,
    max_value=600,

    return_allowlist=True,
)

print("df_fbg shape:", df_fbg.shape)
print("\nTop units:")
print(df_fbg["unit_concept_name"].value_counts(dropna=False).head(15))
print("\nValue distribution:")
print(df_fbg["value_as_number"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("\nTop concepts:")
print(df_fbg["standard_concept_name"].value_counts().head(10))


In [ ]:
# Save
save_phenotypes(df_fbg, bucket, "aou_fbg_labs.csv")


In [ ]:
# Diabetic-range fasting glucose (ADA)
diabetic_fbg = df_fbg[df_fbg["value_as_number"] >= 126].copy()


In [ ]:
first_diabetic_fbg = (
    diabetic_fbg
    .sort_values(["person_id", "measurement_datetime"])
    .groupby("person_id")
    .first()
    .reset_index()
    .loc[:, ["person_id", "value_as_number", "measurement_datetime"]]
    .rename(columns={
        "value_as_number": "first_diabetic_fbg",
        "measurement_datetime": "first_diabetic_fbg_date"
    })
)


In [ ]:
# Save
save_phenotypes(first_diabetic_fbg, bucket, "person_with_diabetic_fasting_glucose.csv")


## Random Blood Glucose


In [ ]:
glucose_concepts = discover_lab_concepts_by_name(
    pattern=r"\bglucose\b",
    limit=1000,
    use_regex=True,
    include_source_concepts=False
)


In [ ]:
import pandas as pd

rbg_candidates = glucose_concepts.copy()
rbg_candidates["concept_name_lc"] = rbg_candidates["concept_name"].astype(str).str.lower()

# -----------------------------
# 1) Positive inclusion filter (must look like blood glucose)
# -----------------------------
keep_pattern = (
    r"\bglucose\b"
    r".*("
    r"serum|plasma|blood|capillary|venous|arterial|glucometer|test strip|meter|point of care|poc|random|casual|spot"
    r")"
)

rbg_keep = rbg_candidates[
    rbg_candidates["concept_name_lc"].str.contains(keep_pattern, regex=True, na=False)
].copy()

# -----------------------------
# 2) Hard exclusions (remove tolerance/challenge/timepoint/body-fluid/derived tests)
# -----------------------------
exclude_pattern = (
    r"\bfasting\b|"                     # fasting labs
    r"post|post[- ]?prandial|meal|"      # post-meal / prandial
    r"ogtt|tolerance|challenge|dose|"    # tolerance testing / challenge / post-dose
    r"\bpre\b|pre[- ]?meal|pre[- ]?dose|"# pre-dose / pre-meal (challenge series)
    r"baseline|specimen|dwell|"          # baseline/specimen series/dwell specimens
    r"\b(15|30|45|60|90|120)\b\s*minute|"# timepoints
    r"\b(1|2|3|4|5|6|7)\b\s*hour|"       # hour timepoints
    r"\b50\s*g\b|\b75\s*g\b|\b100\s*g\b|"# glucose load grams
    r"\blactose\b|\bglucose po\b|"       # oral load keywords
    r"csf|cerebral spinal|synovial|pleural|peritoneal|pericardial|dialysis|body fluid|fluid|"  # non-blood
    r"\burine\b|urinalysis|qualitative|presence|"  # urine/qualitative
    r"g6p|g-6-|glucose-6|dehydrogenase|"  # G6PD, not glucose conc
    r"estimated from glycated hemoglobin|estimated from|eag|mean value|"  # derived from A1c
    r"\binsulin\b|"                      # insulin tests
    r"interpretation|ratio|correlation|difference" # not actual glucose concentration
)

rbg_allowlist_df = rbg_keep[
    ~rbg_keep["concept_name_lc"].str.contains(exclude_pattern, regex=True, na=False)
].copy()

# keep only concepts that are actually used
rbg_allowlist_df = rbg_allowlist_df.sort_values(["n_people", "n_rows"], ascending=False).reset_index(drop=True)

print("RBG allowlist concepts:", rbg_allowlist_df.shape)
display_cols = ["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]
print(rbg_allowlist_df[display_cols].head(30).to_string(index=False))


In [ ]:
# -----------------------------
# Final cleanup: remove device/meta and non-glucose-analyte concepts
# -----------------------------
rbg_allowlist_df = rbg_allowlist_df.copy()
rbg_allowlist_df["concept_name_lc"] = rbg_allowlist_df["concept_name"].astype(str).str.lower()

exclude_more = (
    r"meter device|device panel|vendor|serial number|type of|"   # device/meta concepts
    r"corrected for glucose|"                                    # sodium corrected for glucose
    r"phosphate isomerase|"                                      # enzyme, not glucose concentration
    r"\benzymatic activity\b|"                                   # enzyme-type analytes
    r"\bpanel\b|"                                                # panels
    r"\bratio\b|\bcorrelation\b|\bvendor\b"                      # misc metadata
)

rbg_allowlist_df = rbg_allowlist_df[
    ~rbg_allowlist_df["concept_name_lc"].str.contains(exclude_more, regex=True, na=False)
].copy()

# Recheck
print("RBG allowlist concepts after final cleanup:", rbg_allowlist_df.shape)
print(rbg_allowlist_df[["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]]
      .sort_values(["n_people","n_rows"], ascending=False)
      .head(30)
      .to_string(index=False))


In [ ]:
df_rbg, rbg_allow_audit = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=rbg_allowlist_df,
    date_cutoffs=None,

    keep_vocab=("LOINC",),
    keep_concept_classes=("Lab Test",),

    allow_extra_rules=[
        {"vocabulary_id": "SNOMED", "concept_class_id": "Procedure"}
    ],

    conversions=[
        {
            "pattern": r"(mmol|moles/volume)",
            "transform": lambda s: s * 18.0182,
            "new_unit": "mg/dL (converted from mmol/L)",
        }
    ],

    require_units_containing=("mg/dl", "milligram per deciliter"),
    allow_unit_nulls=True,

    min_value=40,
    max_value=800,

    return_allowlist=True,
)

print("df_rbg shape:", df_rbg.shape)
print("\nTop units:")
print(df_rbg["unit_concept_name"].value_counts(dropna=False).head(15))
print("\nValue distribution:")
print(df_rbg["value_as_number"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("\nTop concepts:")
print(df_rbg["standard_concept_name"].value_counts().head(15))


In [ ]:
# Save
save_phenotypes_fast(df_rbg, bucket, "all_random_blood_glucose")


In [ ]:
import pandas as pd

# -----------------------------
# Define diabetic-range RBG
# -----------------------------
diabetic_rbg = df_rbg[df_rbg["value_as_number"] >= 200].copy()

first_diabetic_rbg = (
    diabetic_rbg
    .sort_values(["person_id", "measurement_datetime"])
    .groupby("person_id")
    .first()
    .reset_index()
    .loc[:, ["person_id", "value_as_number", "measurement_datetime"]]
    .rename(columns={
        "value_as_number": "first_diabetic_rbg",
        "measurement_datetime": "first_diabetic_rbg_date"
    })
)

# Ensure datetime
first_diabetic_rbg["first_diabetic_rbg_date"] = pd.to_datetime(
    first_diabetic_rbg["first_diabetic_rbg_date"],
    errors="coerce",
    utc=True
)


In [ ]:
# Save
save_phenotypes(first_diabetic_rbg, bucket, "person_with_diabetic_random_glucose.csv")


## First T2D Evidence

The earliest qualifying date across the four T2D sources is retained as `t2d_first_evidence_date`. When multiple sources occur on the same earliest date, the source label is assigned deterministically in the order ICD, HbA1c, fasting glucose, then random glucose.


In [ ]:
def standardize_date_column(df, candidates, new_name):
    """Rename the first available date column from a list of candidates."""
    for col in candidates:
        if col in df.columns:
            return df.copy().rename(columns={col: new_name})

    raise ValueError(
        f"None of the candidate date columns are present: {candidates}"
    )


t2d_icd = standardize_date_column(
    t2d_icd_earliest,
    ["condition_start_datetime", "t2d_icd_date", "icd_date"],
    "icd_date",
)[["person_id", "icd_date"]]

t2d_a1c = standardize_date_column(
    first_diabetic_a1c,
    ["first_diabetic_a1c_date", "a1c_date"],
    "a1c_date",
)[["person_id", "a1c_date"]]

t2d_fbg = standardize_date_column(
    first_diabetic_fbg,
    [
        "first_diabetic_fasting_glucose_date",
        "first_diabetic_fbg_date",
        "fbg_date",
    ],
    "fbg_date",
)[["person_id", "fbg_date"]]

t2d_rbg = standardize_date_column(
    first_diabetic_rbg,
    [
        "first_diabetic_rbg_date",
        "first_diabetic_random_glucose_date",
        "rbg_date",
    ],
    "rbg_date",
)[["person_id", "rbg_date"]]

raw_t2d = (
    t2d_icd
    .merge(t2d_a1c, on="person_id", how="outer")
    .merge(t2d_fbg, on="person_id", how="outer")
    .merge(t2d_rbg, on="person_id", how="outer")
)

date_cols = [
    "icd_date",
    "a1c_date",
    "fbg_date",
    "rbg_date",
]

for col in date_cols:
    raw_t2d[col] = parse_datetime(raw_t2d[col])

raw_t2d["t2d_first_evidence_date"] = raw_t2d[date_cols].min(axis=1)

source_priority = [
    ("ICD", "icd_date"),
    ("A1c", "a1c_date"),
    ("FBG", "fbg_date"),
    ("RBG", "rbg_date"),
]


def identify_t2d_source(row):
    event_date = row["t2d_first_evidence_date"]

    if pd.isna(event_date):
        return pd.NA

    for label, col in source_priority:
        if pd.notna(row[col]) and row[col] == event_date:
            return label

    return pd.NA


raw_t2d["t2d_first_evidence_source"] = raw_t2d.apply(
    identify_t2d_source,
    axis=1,
)

source_flag_cols = []
for label, col in source_priority:
    flag_col = f"has_{label.lower()}"
    raw_t2d[flag_col] = raw_t2d[col].notna()
    source_flag_cols.append(flag_col)

raw_t2d["t2d_source_count"] = (
    raw_t2d[source_flag_cols]
    .sum(axis=1)
    .astype(int)
)

raw_t2d["classification"] = raw_t2d.apply(
    lambda row: " + ".join(
        label
        for label, col in source_priority
        if pd.notna(row[col])
    ),
    axis=1,
)

final_t2d_dx = raw_t2d[
    [
        "person_id",
        "t2d_first_evidence_date",
        "t2d_first_evidence_source",
        "t2d_source_count",
        "icd_date",
        "a1c_date",
        "fbg_date",
        "rbg_date",
        "classification",
    ]
].copy()

final_t2d_dx = normalize_person_id(
    final_t2d_dx,
    label="final T2D",
)

assert final_t2d_dx["person_id"].is_unique
assert final_t2d_dx["t2d_first_evidence_date"].notna().all()

print(
    f"Participants with qualifying T2D evidence: "
    f"{len(final_t2d_dx):,}"
)

print("\nEarliest T2D evidence source:")
print(
    final_t2d_dx["t2d_first_evidence_source"]
    .value_counts(dropna=False)
)

save_phenotypes(
    final_t2d_dx,
    bucket,
    "final_t2d_first_evidence.csv",
)


# 4. Kidney Disease Ascertainment

Kidney disease evidence is curated from diagnosis codes, persistent albuminuria, persistent reduced eGFR, dialysis, and kidney transplantation. Each phenotype retains a dated qualifying event for downstream cohort construction.


## Condition Extraction Functions


In [ ]:
from google.cloud import bigquery
import pandas as pd

def subset_code_exact_match_for_terms(sql_query, search_terms, id_column, condition_start_datetime):
    # Set up the BigQuery client
    client = bigquery.Client()

    # Convert the search terms to lowercase and create a comma-separated string
    lower_search_terms = [term.lower() for term in search_terms]
    search_conditions = ' OR '.join([f"LOWER(source_concept_code) = '{term}'" for term in lower_search_terms])

    # Define the query to subset the rows based on the exact match condition for multiple terms
    subset_query = f"""
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY {id_column} ORDER BY {condition_start_datetime} DESC) AS row_num
        FROM (
            {sql_query}
        )
        WHERE {search_conditions}
    )
    """

    # Execute the query and retrieve the results directly as a Pandas DataFrame
    subset_df = client.query(subset_query).to_dataframe()
        
    return subset_df


In [ ]:
import pandas as pd

def subset_unique_earliest_date(dataframe, id_column, date_column):
    idx = dataframe.groupby(id_column)[date_column].idxmin()
    return dataframe.loc[idx].reset_index(drop=True)


In [ ]:
dataset_all_condition_sql = f"""
SELECT
    c.person_id,
    c.condition_start_datetime,
    c.condition_end_datetime,
    c_standard.concept_name  AS standard_concept_name,
    c_source.concept_name    AS source_concept_name,
    c_source.concept_code    AS source_concept_code,
    c_source.vocabulary_id   AS source_vocabulary,
    c_type.concept_name      AS condition_type,
    visit.concept_name       AS visit_type,
    c_status.concept_name    AS condition_status
FROM `{CDR_DATASET}.condition_occurrence` c
LEFT JOIN `{CDR_DATASET}.concept` c_standard
  ON c.condition_concept_id = c_standard.concept_id
LEFT JOIN `{CDR_DATASET}.concept` c_source
  ON c.condition_source_concept_id = c_source.concept_id
LEFT JOIN `{CDR_DATASET}.concept` c_type
  ON c.condition_type_concept_id = c_type.concept_id
LEFT JOIN `{CDR_DATASET}.visit_occurrence` v
  ON c.visit_occurrence_id = v.visit_occurrence_id
LEFT JOIN `{CDR_DATASET}.concept` visit
  ON v.visit_concept_id = visit.concept_id
LEFT JOIN `{CDR_DATASET}.concept` c_status
  ON c.condition_status_concept_id = c_status.concept_id
"""


## CKD Diagnosis Codes

General CKD diagnosis codes are retained for descriptive and quality-control use. They do not independently define the DKD outcome in the cohort-construction notebook.


In [ ]:
ckd_codes = [
    # ICD10:
    'N18', 'N18.1', 'N18.2', 'N18.3', 'N18.30', 'N18.31', 'N18.32', 'N18.4', 'N18.5', 'N18.6', 'N18.9',
    # ICD9:
    '585', '585.1', '585.2', '585.3', '585.4', '585.5', '585.6', '585.9'
]


In [ ]:
ckd_icd = subset_code_exact_match_for_terms(dataset_all_condition_sql, ckd_codes, 'person_id', 'condition_start_datetime')


In [ ]:
ckd_icd_earliest = subset_unique_earliest_date(ckd_icd, 'person_id', 'condition_start_datetime')


In [ ]:
# Save
save_phenotypes(ckd_icd_earliest, bucket, "ckd_dignosis_icd_codes.csv")


## DKD Diagnosis Codes

DKD-specific diagnosis codes identify diabetic nephropathy or diabetic chronic kidney disease.


In [ ]:
dkd_codes_t2d = [
    # --------------------
    # ICD-10-CM
    # --------------------
    "E11.21",  # Type 2 diabetes mellitus with diabetic nephropathy
    "E11.22",  # Type 2 diabetes mellitus with diabetic chronic kidney disease

    # --------------------
    # ICD-9-CM (legacy)
    # --------------------
    "250.40",  # Diabetes with renal manifestations, type II or unspecified
    "250.42",  # Diabetes with renal manifestations, type II uncontrolled
]


In [ ]:
dkd_icd = subset_code_exact_match_for_terms(dataset_all_condition_sql, dkd_codes_t2d, 'person_id', 'condition_start_datetime')


In [ ]:
dkd_icd_earliest = subset_unique_earliest_date(dkd_icd, 'person_id', 'condition_start_datetime')


In [ ]:
# Save
save_phenotypes(dkd_icd_earliest, bucket, "dkd_dignosis_icd_codes.csv")


## Albuminuria

Reported and computed urine albumin-to-creatinine ratio (UACR) measurements are normalized to mg/g. Persistent albuminuria is defined by two qualifying UACR values at least 90 days apart.


### Reported UACR


In [ ]:
uacr_concepts = discover_lab_concepts_by_name(
    pattern=r"albumin.*creatinine|alb.*creat|uacr|acr\b|microalbumin.*creatinine|urine albumin.*creatinine",
    limit=300,
    use_regex=True,
    include_source_concepts=False
)

uacr_concepts[["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]].head(40)


In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import os

def extract_reported_uacr_from_discovered_concepts(
    concepts_df: pd.DataFrame,
    workspace_cdr: str | None = None,
    date_cutoffs: pd.DataFrame | None = None,   # optional: columns person_id, cutoff_date
    keep_vocab: tuple[str, ...] = ("LOINC", "SNOMED"),
    keep_concept_classes: tuple[str, ...] = ("Lab Test",),
    allow_snomed_procedure: bool = True,
    drop_name_contains: tuple[str, ...] = ("panel", "renal function panel"),  # avoid panels
    require_units_containing: tuple[str, ...] = (),  # UACR often missing unit in AoU; default no unit filter
    min_value: float = 0.0,
    max_value: float = 20000.0,
    albuminuria_threshold_mg_g: float = 30.0,
    return_allowlist: bool = True,
):
    """
    Extract *reported* UACR (albumin/creatinine ratio) measurements from OMOP measurement table
    using a curated allowlist derived from a discovery table.

    Output includes:
      - df_uacr_clean: cleaned rows with uacr_mg_g
      - first_albuminuria_uacr: earliest date per person with UACR >= threshold
      - allowlist_df: audit table of included concepts (optional)
    """
    client = bigquery.Client()
    workspace = workspace_cdr or CDR_DATASET

    # -----------------------
    # 1) Curate allowlist
    # -----------------------
    dfc = concepts_df.copy()
    dfc["concept_name_lc"] = dfc["concept_name"].astype(str).str.lower()

    for bad in drop_name_contains:
        dfc = dfc[~dfc["concept_name_lc"].str.contains(str(bad).lower(), na=False)]

    allow = dfc[
        dfc["vocabulary_id"].isin(keep_vocab) &
        dfc["concept_class_id"].isin(keep_concept_classes)
    ].copy()

    if allow_snomed_procedure:
        allow_snomed = dfc[
            (dfc["vocabulary_id"] == "SNOMED") &
            (dfc["concept_class_id"] == "Procedure")
        ].copy()
        allow = pd.concat([allow, allow_snomed], ignore_index=True)

    allow_ids = (
        pd.to_numeric(allow["measurement_concept_id"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if not allow_ids:
        raise ValueError("No UACR concept_ids selected. Check discovery output / filters.")

    # -----------------------
    # 2) Optional cutoff join
    # -----------------------
    cutoff_join_sql = ""
    where_cutoff = ""
    cutoff_table = None

    if date_cutoffs is not None:
        cut = date_cutoffs.copy()
        cut["cutoff_date"] = pd.to_datetime(cut["cutoff_date"], errors="coerce", utc=True)
        cut = cut.dropna(subset=["person_id", "cutoff_date"])
        cut["person_id"] = cut["person_id"].astype(int)

        cutoff_table = f"{workspace}._tmp_uacr_cutoffs"
        client.load_table_from_dataframe(cut, cutoff_table).result()

        cutoff_join_sql = f"JOIN `{cutoff_table}` cut ON m.person_id = cut.person_id"
        where_cutoff = "AND m.measurement_datetime <= cut.cutoff_date"

    # -----------------------
    # 3) Pull measurements
    # -----------------------
    sql = f"""
    SELECT
      m.person_id,
      m.measurement_concept_id,
      m.measurement_datetime,
      m.value_as_number,
      m.unit_concept_id,
      c.concept_name AS standard_concept_name,
      c.vocabulary_id AS standard_vocabulary,
      c.concept_class_id AS standard_concept_class,
      u.concept_name AS unit_concept_name
    FROM `{workspace}.measurement` m
    JOIN `{workspace}.concept` c
      ON m.measurement_concept_id = c.concept_id
    LEFT JOIN `{workspace}.concept` u
      ON m.unit_concept_id = u.concept_id
    {cutoff_join_sql}
    WHERE m.measurement_concept_id IN UNNEST(@concept_ids)
      AND m.measurement_datetime IS NOT NULL
      AND m.value_as_number IS NOT NULL
      {where_cutoff}
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("concept_ids", "INT64", allow_ids)]
    )
    df_uacr = client.query(sql, job_config=job_config).to_dataframe()

    if cutoff_table is not None:
        client.delete_table(cutoff_table, not_found_ok=True)

    # -----------------------
    # 4) Cleaning / QC
    # -----------------------
    df_uacr["person_id"] = df_uacr["person_id"].astype(int)
    df_uacr["measurement_datetime"] = pd.to_datetime(df_uacr["measurement_datetime"], errors="coerce", utc=True)
    df_uacr["value_as_number"] = pd.to_numeric(df_uacr["value_as_number"], errors="coerce")
    df_uacr["unit_concept_name"] = df_uacr["unit_concept_name"].astype("string")

    df_uacr = df_uacr.dropna(subset=["measurement_datetime", "value_as_number"])
    df_uacr = df_uacr[df_uacr["value_as_number"] != 10000000.0]  # sentinel

    # Unit filtering (off by default for AoU UACR due to many missing units)
    if require_units_containing:
        unit_mask = False
        for tok in require_units_containing:
            unit_mask = unit_mask | df_uacr["unit_concept_name"].str.contains(str(tok), case=False, na=False)
        df_uacr = df_uacr[unit_mask]

    # Range filter
    df_uacr = df_uacr[(df_uacr["value_as_number"] >= min_value) & (df_uacr["value_as_number"] <= max_value)]

    # Standardized column
    df_uacr_clean = df_uacr.copy()
    df_uacr_clean["uacr_mg_g"] = df_uacr_clean["value_as_number"]

    # -----------------------
    # 5) First albuminuria date (reported UACR only)
    # -----------------------
    uacr_albuminuria = df_uacr_clean[df_uacr_clean["uacr_mg_g"] >= albuminuria_threshold_mg_g].copy()

    first_albuminuria_uacr = (
        uacr_albuminuria.sort_values(["person_id", "measurement_datetime"])
        .groupby("person_id", as_index=False)
        .first()
        .loc[:, ["person_id", "uacr_mg_g", "measurement_datetime"]]
        .rename(columns={
            "uacr_mg_g": "first_albuminuria_uacr_mg_g",
            "measurement_datetime": "first_albuminuria_uacr_date",
        })
    )

    # -----------------------
    # 6) Nice printout
    # -----------------------
    n_rows = len(df_uacr_clean)
    n_people_any = df_uacr_clean["person_id"].nunique()
    n_people_alb = first_albuminuria_uacr["person_id"].nunique()

    print("\n" + "="*72)
    print("Reported UACR Extraction Summary")
    print("="*72)
    print(f"Rows extracted (reported UACR):            {n_rows:,}")
    print(f"People with any reported UACR:             {n_people_any:,}")
    print(f"People with albuminuria (UACR ≥ {albuminuria_threshold_mg_g:g}): {n_people_alb:,}")
    print("-"*72)

    # Units snapshot
    top_units = df_uacr["unit_concept_name"].value_counts(dropna=False).head(10)
    print("Top units (raw, top 10):")
    for k, v in top_units.items():
        key = "<NA>" if pd.isna(k) else str(k)
        print(f"  {key:<35} {int(v):,}")

    # Distribution snapshot
    desc = df_uacr_clean["uacr_mg_g"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    print("-"*72)
    print("UACR mg/g distribution:")
    print(desc.to_string())
    print("="*72)

    # Date summary for albuminuria
    if len(first_albuminuria_uacr) > 0:
        d = pd.to_datetime(first_albuminuria_uacr["first_albuminuria_uacr_date"], utc=True)
        qs = d.quantile([0.25, 0.5, 0.75])
        print("\n" + "="*72)
        print(f"First Albuminuria by Reported UACR (≥ {albuminuria_threshold_mg_g:g} mg/g) Date Summary")
        print("="*72)
        print(f"Earliest:  {d.min().date()}")
        print(f"25th pct:  {qs.loc[0.25].date()}")
        print(f"Median:    {qs.loc[0.5].date()}")
        print(f"75th pct:  {qs.loc[0.75].date()}")
        print(f"Latest:    {d.max().date()}")
        print("="*72)

    # Allowlist audit table
    allowlist_df = allow.sort_values(["n_people", "n_rows"], ascending=False).reset_index(drop=True)

    if return_allowlist:
        return df_uacr_clean, first_albuminuria_uacr, allowlist_df
    return df_uacr_clean, first_albuminuria_uacr


# -----------------------
# usage
# -----------------------
df_uacr_clean, first_albuminuria_uacr, uacr_allow = extract_reported_uacr_from_discovered_concepts(
    concepts_df=uacr_concepts,     # output of discover_lab_concepts_by_name(...)
    date_cutoffs=None,
    keep_vocab=("LOINC", "SNOMED"),
    keep_concept_classes=("Lab Test",),  # keep strict; SNOMED procedures added separately
    allow_snomed_procedure=True,
    drop_name_contains=("panel", "renal function panel"),
    require_units_containing=(),         # do NOT filter by unit in AoU
    min_value=0.0,
    max_value=20000.0,
    albuminuria_threshold_mg_g=30.0,
    return_allowlist=True,
)


In [ ]:
import pandas as pd
import numpy as np

def normalize_reported_uacr_units(
    df_reported_uacr_raw: pd.DataFrame,
    value_col: str = "value_as_number",
    unit_col: str = "unit_concept_name",
    out_col: str = "reported_uacr_mg_g",
    drop_if_unit_unknown: bool = False,
    cap_mg_g: float = 20000.0,   # optional plausibility cap
):
    """
    Normalize REPORTED UACR values to mg/g using unit_concept_name only.

    Supported units:
      - mg/g variants: kept as-is
      - ug/mg variants: numeric 1:1 with mg/g
      - mg/mmol variants: convert using creatinine MW:
           mg/g = (mg/mmol) / 8.84   [since 1 mmol creatinine = 113.12 mg; 1000/113.12 = 8.84]

    Everything else (including missing/No matching concept/no value/ratio) -> NaN.
    """
    df = df_reported_uacr_raw.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df[unit_col] = df[unit_col].astype("string")

    unit_norm = (
        df[unit_col]
        .fillna("<NA>")
        .str.strip()
        .str.lower()
    )
    df["reported_uacr_unit_norm"] = unit_norm

    df[out_col] = np.nan
    df["reported_uacr_unit_rule"] = pd.NA

    # hard exclude "unknown-ish" unit labels up front
    unknownish = (
        unit_norm.isin(["<na>", "no matching concept", "no value"]) |
        unit_norm.eq("ratio") |
        unit_norm.eq("second") |
        unit_norm.eq("times")
    )

    # ---- Rule A: mg/g variants ----
    mg_g_mask = (~unknownish) & (
        unit_norm.str.contains("milligram per gram", na=False) |
        unit_norm.str.contains(r"\bmg/g\b", na=False) |
        unit_norm.str.contains("milligram per gram of creatinine", na=False)
    )
    df.loc[mg_g_mask, out_col] = df.loc[mg_g_mask, value_col]
    df.loc[mg_g_mask, "reported_uacr_unit_rule"] = "mg/g (as-is)"

    # ---- Rule B: ug/mg variants (1:1 with mg/g) ----
    ug_mg_mask = (~unknownish) & (
        unit_norm.str.contains("microgram per milligram", na=False) |
        unit_norm.str.contains("microgram per milligram of creatinine", na=False) |
        unit_norm.str.contains(r"\bmcg/mg\b", na=False) |
        unit_norm.str.contains(r"\bug/mg\b", na=False)
    )
    df.loc[ug_mg_mask, out_col] = df.loc[ug_mg_mask, value_col]
    df.loc[ug_mg_mask, "reported_uacr_unit_rule"] = "ug/mg → mg/g (1:1)"

    # ---- Rule C: mg/mmol (convert) ----
    mg_mmol_mask = (~unknownish) & (
        unit_norm.str.contains("milligram per millimole", na=False) |
        unit_norm.str.contains("milligram per millimole of creatinine", na=False)
    )
    df.loc[mg_mmol_mask, out_col] = df.loc[mg_mmol_mask, value_col] / 8.84
    df.loc[mg_mmol_mask, "reported_uacr_unit_rule"] = "mg/mmol → mg/g (/8.84)"

    # ---- Mark anything else as unknown/unsupported ----
    unknown_mask = df[out_col].isna() & df[value_col].notna()
    df.loc[unknown_mask, "reported_uacr_unit_rule"] = "unknown/unsupported unit"

    # ---- Plausibility cap ----
    bad = df[out_col].notna() & ((df[out_col] < 0) | (df[out_col] > cap_mg_g))
    df.loc[bad, out_col] = np.nan
    df.loc[bad, "reported_uacr_unit_rule"] = f"out of range (<0 or >{cap_mg_g:g})"

    if drop_if_unit_unknown:
        df = df.dropna(subset=[out_col])

    return df


def summarize_first_reported_albuminuria_uacr(
    df_reported_uacr_norm: pd.DataFrame,
    threshold_mg_g: float = 30.0,
    person_col: str = "person_id",
    date_col: str = "measurement_datetime",
    uacr_col: str = "reported_uacr_mg_g",
):
    df = df_reported_uacr_norm.copy()
    df[person_col] = df[person_col].astype("Int64")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df[uacr_col] = pd.to_numeric(df[uacr_col], errors="coerce")

    df_use = df.dropna(subset=[person_col, date_col, uacr_col]).copy()

    alb = df_use[df_use[uacr_col] >= threshold_mg_g].copy()
    first = (
        alb.sort_values([person_col, date_col])
        .groupby(person_col, as_index=False)
        .first()
        .loc[:, [person_col, uacr_col, date_col]]
        .rename(columns={
            uacr_col: "first_reported_uacr_mg_g",
            date_col: "first_reported_uacr_date",
        })
    )

    print("\n" + "="*72)
    print("Reported UACR Normalization + Albuminuria Summary")
    print("="*72)
    print(f"Rows input (reported UACR):                   {len(df_reported_uacr_norm):,}")
    print(f"People with any reported UACR (input):        {df_reported_uacr_norm[person_col].nunique():,}")
    print(f"Rows with usable normalized reported UACR:    {len(df_use):,}")
    print(f"People with usable normalized reported UACR:  {df_use[person_col].nunique():,}")
    print(f"People with albuminuria (≥{threshold_mg_g:g} mg/g):             {first[person_col].nunique():,}")
    print("-"*72)

    audit = (
        df.assign(is_convertible=df[uacr_col].notna())
        .groupby("reported_uacr_unit_norm", dropna=False)
        .agg(
            n_rows=("reported_uacr_unit_norm", "size"),
            n_people=(person_col, "nunique"),
            pct_convertible=("is_convertible", "mean"),
            median_raw=(value_col := "value_as_number", "median"),
            median_mg_g=(uacr_col, "median"),
            p95_mg_g=(uacr_col, lambda s: np.nanpercentile(s, 95)),
            max_mg_g=(uacr_col, "max"),
        )
        .reset_index()
        .sort_values("n_rows", ascending=False)
    )
    print("Unit normalization audit (top 15 by rows):")
    print(audit.head(15).to_string(index=False))

    if len(first) > 0:
        d = pd.to_datetime(first["first_reported_uacr_date"], utc=True)
        qs = d.quantile([0.25, 0.5, 0.75])
        print("-"*72)
        print("First albuminuria date distribution:")
        print(f"  Earliest:  {d.min().date()}")
        print(f"  25th pct:  {qs.loc[0.25].date()}")
        print(f"  Median:    {qs.loc[0.5].date()}")
        print(f"  75th pct:  {qs.loc[0.75].date()}")
        print(f"  Latest:    {d.max().date()}")
        print("="*72)

    return first, audit


# =========================
# Usage
# =========================
df_reported_uacr_norm = normalize_reported_uacr_units(
    df_uacr_clean,
    drop_if_unit_unknown=False
)

df_first_albuminuria_reported_uacr, reported_uacr_unit_audit = summarize_first_reported_albuminuria_uacr(
    df_reported_uacr_norm,
    threshold_mg_g=30.0
)


In [ ]:
# Save
save_phenotypes(df_reported_uacr_norm, bucket, "aou_reported_uacr.csv")


### Computed UACR

Computed UACR is derived from paired urine albumin and urine creatinine measurements recorded on the same calendar date.


In [ ]:
ucreat_concepts = discover_lab_concepts_by_name(
    pattern=r"(urine|urinary|24\s*hour).*creatinine|creatinine.*(urine|urinary|24\s*hour)|\bucreat\b",
    limit=300,
    use_regex=True,
    include_source_concepts=False
)


In [ ]:
ualb_concepts = discover_lab_concepts_by_name(
    pattern=r"(urine|urinary).*albumin|albumin.*(urine|urinary)|\bualb\b",
    limit=300,
    use_regex=True,
    include_source_concepts=False
)


In [ ]:
import pandas as pd
import numpy as np
import re

def build_allowlist_from_concepts(
    concepts_df: pd.DataFrame,
    concept_id_col: str = "measurement_concept_id",
    concept_name_col: str = "concept_name",
    vocabulary_col: str = "vocabulary_id",
    concept_class_col: str = "concept_class_id",
    keep_vocab=("LOINC","SNOMED"),
    keep_concept_classes=("Lab Test","Procedure"),
    must_contain=(),
    must_not_contain=(),
    exact_allow_names=None,
):
    df = concepts_df.copy()

    # basic filters
    df = df[df[vocabulary_col].isin(keep_vocab)]
    df = df[df[concept_class_col].isin(keep_concept_classes)]

    name = df[concept_name_col].astype(str)

    # must contain (all)
    for s in must_contain:
        df = df[name.str.contains(s, case=False, na=False)]
        name = df[concept_name_col].astype(str)

    # must NOT contain (any)
    if must_not_contain:
        pat = "|".join([re.escape(s) for s in must_not_contain])
        df = df[~name.str.contains(pat, case=False, na=False)]

    # optional exact allowlist by concept_name
    if exact_allow_names is not None:
        df = df[df[concept_name_col].isin(set(exact_allow_names))]

    # return only the concept rows
    return df.sort_values(["n_people","n_rows"], ascending=False)


In [ ]:
# ---- 1) build concept allowlist (albumin concentration only) ----
UALB_EXACT = {
    "Microalbumin [Mass/volume] in Urine",
    "Albumin [Mass/volume] in Urine",
    "Microalbumin [Mass/volume] in 24 hour Urine",
    "Albumin [Mass/volume] in 24 hour Urine",
    "Microalbumin [Mass/volume] in 12 hour Urine",
}

ualb_allow_concepts = build_allowlist_from_concepts(
    concepts_df=ualb_concepts,   # <- your discovered albumin concepts df
    keep_vocab=("LOINC","SNOMED"),
    keep_concept_classes=("Lab Test","Procedure"),
    must_contain=("Urine",),     # urine only
    must_not_contain=(
        "Creatinine", "Ratio", "Mass Ratio", "Molar ratio",
        "panel", "profile",
        "Presence", "Interpretation",
        "Test strip", "semiquantitative",
        "Electrophoresis",
        "clearance", "renal clearance",
        "Albumin/Globulin",
        "Protein.total", "Pure mass fraction",
        "Renal function panel", "Hepatic function panel",
        "Serum", "Plasma", "Blood", "CSF", "Cerebrospinal", "Body fluid"
    ),
    exact_allow_names=UALB_EXACT,  # tight final filter
)

print("UALB allow concepts:", ualb_allow_concepts.shape)
display(ualb_allow_concepts[[ "measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]].head(30))

# ---- 2) extract from measurement table using your existing function ----
df_ualb_raw, ualb_allowlist = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=ualb_allow_concepts,
    date_cutoffs=None,
    keep_vocab=("LOINC","SNOMED"),
    keep_concept_classes=("Lab Test","Procedure"),
    drop_name_contains=(),              # already filtered; keep empty or add extra drops
    require_units_containing=None,      # don't require (AoU units messy)
    allow_unit_nulls=True,
    min_value=0.0,
    max_value=100000.0,
    return_allowlist=True,
)

print("df_ualb_raw:", df_ualb_raw.shape, "people:", df_ualb_raw["person_id"].nunique())
print(df_ualb_raw["standard_concept_name"].value_counts().head(20))
print(df_ualb_raw["unit_concept_name"].value_counts().head(15))


In [ ]:
# --- Creatinine in urine allowlist (LOINC + SNOMED) ---

ucr_allow_concepts = build_allowlist_from_concepts(
    concepts_df=ucreat_concepts,
    keep_vocab=("LOINC", "SNOMED"),
    keep_concept_classes=("Lab Test", "Procedure"),
    must_contain=("Creatinine", "Urine"),
    must_not_contain=(
        # ratios of any kind
        "Ratio", "Mass Ratio", "Molar ratio",

        # not desired for our "raw urine creatinine" extraction
        "clearance", "renal clearance",
        "Interpretation", "acceptable",

        # panels / composites
        "panel", "profile",

        # other analytes / composites
        "Protein", "Urea", "Calcium", "Sodium", "Potassium",
        "Cortisol", "Metanephrine", "Normetanephrine", "Catecholamines",
        "Vanillylmandelate", "Hydroxyindoleacetate", "Porphyrin",

        # tox / drug screens (optional)
        "THC", "Tetrahydrocannabinol", "amphetamine", "benzodiazepine", "opioid",
    ),
)

print("UCR allow concepts:", ucr_allow_concepts.shape)
print(ucr_allow_concepts["vocabulary_id"].value_counts(dropna=False))
display(
    ucr_allow_concepts[
        ["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]
    ].head(30)
)


df_ucr_raw, ucr_allowlist = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=ucr_allow_concepts,
    date_cutoffs=None,
    keep_vocab=("LOINC","SNOMED"),
    keep_concept_classes=("Lab Test","Procedure"),
    drop_name_contains=(),
    require_units_containing=None,
    allow_unit_nulls=True,
    min_value=0.0,
    max_value=200000.0,
    return_allowlist=True,
)

print("df_ucr_raw:", df_ucr_raw.shape, "people:", df_ucr_raw["person_id"].nunique())
print("\nTop concepts:")
print(df_ucr_raw["standard_concept_name"].value_counts().head(20))
print("\nTop units:")
print(df_ucr_raw["unit_concept_name"].value_counts().head(20))


In [ ]:
# ----------------------------
# Keep only spot urine units
# ----------------------------

UCR_KEEP_UNITS = {
    "milligram per deciliter", "mg/dL",
    "milligram per liter", "mg/L",
    "milligram per milliliter",
    "micromole per liter", "µmol/L",
    "No matching concept"
}

UALB_KEEP_UNITS = {
    "milligram per deciliter", "mg/dL",
    "milligram per liter", "mg/L",
    "microgram per milliliter",
    "milligram per milliliter",
    "No matching concept"
}

df_ucr_spot = df_ucr_raw[df_ucr_raw["unit_concept_name"].isin(UCR_KEEP_UNITS)].copy()
df_ualb_spot = df_ualb_raw[df_ualb_raw["unit_concept_name"].isin(UALB_KEEP_UNITS)].copy()

print("Creatinine spot:", df_ucr_spot.shape, df_ucr_spot["person_id"].nunique())
print("Albumin spot:", df_ualb_spot.shape, df_ualb_spot["person_id"].nunique())


In [ ]:
def normalize_ucr_to_mg_dl(df):
    df = df.copy()
    u = df["unit_concept_name"].str.lower()

    df["ucr_mg_dl"] = np.nan

    df.loc[u.isin(["mg/dl", "milligram per deciliter", "no matching concept"]),
           "ucr_mg_dl"] = df["value_as_number"]

    df.loc[u.isin(["mg/l", "milligram per liter"]),
           "ucr_mg_dl"] = df["value_as_number"] / 10.0

    df.loc[u.isin(["milligram per milliliter"]),
           "ucr_mg_dl"] = df["value_as_number"] * 100.0

    df.loc[u.isin(["micromole per liter", "µmol/l"]),
           "ucr_mg_dl"] = df["value_as_number"] * 0.0113

    return df

df_ucr_spot_norm = normalize_ucr_to_mg_dl(df_ucr_spot)


In [ ]:
def normalize_ualb_to_mg_dl(df):
    df = df.copy()
    u = df["unit_concept_name"].str.lower()

    df["ualb_mg_dl"] = np.nan

    df.loc[u.isin(["mg/dl", "milligram per deciliter", "no matching concept"]),
           "ualb_mg_dl"] = df["value_as_number"]

    df.loc[u.isin(["mg/l", "milligram per liter"]),
           "ualb_mg_dl"] = df["value_as_number"] / 10.0

    df.loc[u.isin(["microgram per milliliter"]),
           "ualb_mg_dl"] = df["value_as_number"] / 1000.0

    df.loc[u.isin(["milligram per milliliter"]),
           "ualb_mg_dl"] = df["value_as_number"] * 100.0

    return df

df_ualb_spot_norm = normalize_ualb_to_mg_dl(df_ualb_spot)


In [ ]:
df_ucr_spot_norm = df_ucr_spot_norm.query("5 <= ucr_mg_dl <= 500")
df_ualb_spot_norm = df_ualb_spot_norm.query("0 <= ualb_mg_dl <= 2000")

print("Creatinine after bounds:", df_ucr_spot_norm.shape)
print("Albumin after bounds:", df_ualb_spot_norm.shape)


In [ ]:
# Save normalized spot urine albumin for downstream feature construction.
save_gcs_parquet(
    df_ualb_spot_norm[
        [
            "person_id",
            "measurement_datetime",
            "ualb_mg_dl",
        ]
    ].copy(),
    f"{CURATION_DIR}/longitudinal_spot_urine_albumin.parquet",
)


In [ ]:
df_ucr_spot_norm["date"]  = df_ucr_spot_norm["measurement_datetime"].dt.date
df_ualb_spot_norm["date"] = df_ualb_spot_norm["measurement_datetime"].dt.date

df_uacr_computed_pairs = (
    df_ualb_spot_norm
    .merge(
        df_ucr_spot_norm,
        on=["person_id", "date"],
        suffixes=("_alb", "_cr"),
        how="inner"
    )
)

print("Computed UACR pairs:", df_uacr_computed_pairs.shape)
print("People with computed UACR:", df_uacr_computed_pairs["person_id"].nunique())


In [ ]:
df_uacr_computed_pairs["uacr_computed_mg_g"] = (
    df_uacr_computed_pairs["ualb_mg_dl"]
    / df_uacr_computed_pairs["ucr_mg_dl"]
) * 1000.0


In [ ]:
df_uacr_computed = (
    df_uacr_computed_pairs
    .query("0 <= uacr_computed_mg_g <= 20000")
    .copy()
)

df_uacr_computed["uacr_computed_mg_g"].describe(
    percentiles=[0.5, 0.95, 0.99]
)


In [ ]:
# Save
save_phenotypes(df_uacr_computed, bucket, "aou_computed_uacr.csv")


In [ ]:
df_uacr_computed["albuminuria_computed"] = (
    df_uacr_computed["uacr_computed_mg_g"] >= 30
).astype(int)


In [ ]:
first_albuminuria_uacr_computed = (
    df_uacr_computed[df_uacr_computed["albuminuria_computed"] == 1]
    .sort_values("measurement_datetime_alb")
    .groupby("person_id", as_index=False)
    .first()[["person_id", "measurement_datetime_alb", "uacr_computed_mg_g"]]
    .rename(columns={
        "measurement_datetime_alb": "first_albuminuria_uacr_computed_date"
    })
)

print("People with computed albuminuria:",
      first_albuminuria_uacr_computed.shape[0])


### Persistent Albuminuria


In [ ]:
import pandas as pd
import numpy as np

def first_persistent_albuminuria(
    df_uacr: pd.DataFrame,
    uacr_col: str,
    threshold_mg_g: float = 30.0,
    min_days_apart: int = 90,
    person_col: str = "person_id",
    date_col: str = "measurement_datetime",
    out_prefix: str = "reported",  # or "computed"
) -> pd.DataFrame:
    df = df_uacr.copy()

    df[person_col] = pd.to_numeric(df[person_col], errors="coerce").astype("Int64")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df[uacr_col] = pd.to_numeric(df[uacr_col], errors="coerce")

    df = df.dropna(subset=[person_col, date_col, uacr_col]).sort_values([person_col, date_col])

    # keep only qualifying highs
    q = df.loc[df[uacr_col] >= threshold_mg_g, [person_col, date_col, uacr_col]].copy()
    if q.empty:
        return pd.DataFrame(columns=[
            person_col,
            f"first_qualifying_albuminuria_{out_prefix}_date",
            f"first_persistent_albuminuria_{out_prefix}_date",
            f"confirming_uacr_{out_prefix}_mg_g",
        ])

    min_delta = np.timedelta64(min_days_apart, "D")

    def per_person(g: pd.DataFrame) -> pd.Series:
        # g is already sorted
        dates = g[date_col].to_numpy()
        vals  = g[uacr_col].to_numpy()

        first_date = dates[0]
        # find earliest j where dates[j] - first_date >= min_days_apart
        for j in range(1, len(dates)):
            if (dates[j] - first_date) >= min_delta:
                return pd.Series({
                    f"first_qualifying_albuminuria_{out_prefix}_date": first_date,
                    f"first_persistent_albuminuria_{out_prefix}_date": dates[j],
                    f"confirming_uacr_{out_prefix}_mg_g": vals[j],
                })

        return pd.Series({
            f"first_qualifying_albuminuria_{out_prefix}_date": pd.NaT,
            f"first_persistent_albuminuria_{out_prefix}_date": pd.NaT,
            f"confirming_uacr_{out_prefix}_mg_g": np.nan,
        })

    out = q.groupby(person_col).apply(per_person).reset_index()

    # keep only confirmed
    out = out.dropna(subset=[f"first_persistent_albuminuria_{out_prefix}_date"]).reset_index(drop=True)

    # stable column order
    out = out[[
        person_col,
        f"first_qualifying_albuminuria_{out_prefix}_date",
        f"first_persistent_albuminuria_{out_prefix}_date",
        f"confirming_uacr_{out_prefix}_mg_g",
    ]]
    return out


In [ ]:
first_persist_reported = first_persistent_albuminuria(
    df_reported_uacr_norm,                      # your reported-uacr normalized table
    uacr_col="reported_uacr_mg_g",      # rename your normalized col to this
    date_col= "measurement_datetime",
    out_prefix="reported",
    min_days_apart=90,
)

first_persist_computed = first_persistent_albuminuria(
    df_uacr_computed,                  # your computed-uacr table
    uacr_col="uacr_computed_mg_g",
    date_col= "measurement_datetime_alb",
    out_prefix="computed",
    min_days_apart=90,
)


In [ ]:
# Save
save_phenotypes(first_persist_reported, bucket, "person_with_ckd_reported_uacr.csv")


In [ ]:
# Save
save_phenotypes(first_persist_computed, bucket, "person_with_ckd_computed_uacr.csv")


In [ ]:
either_uacr = (
    first_persist_reported.merge(first_persist_computed, on="person_id", how="outer")
)

either_uacr["first_persistent_albuminuria_any_date"] = either_uacr[
    ["first_persistent_albuminuria_reported_date",
     "first_persistent_albuminuria_computed_date"]
].min(axis=1)

either_uacr["albuminuria_source_any"] = np.where(
    either_uacr["first_persistent_albuminuria_any_date"].eq(either_uacr["first_persistent_albuminuria_reported_date"]),
    "reported",
    "computed"
)

# if one side is missing, fix label
either_uacr.loc[either_uacr["first_persistent_albuminuria_reported_date"].isna(), "albuminuria_source_any"] = "computed"
either_uacr.loc[either_uacr["first_persistent_albuminuria_computed_date"].isna(), "albuminuria_source_any"] = "reported"


In [ ]:
# Save
save_phenotypes(either_uacr, bucket, "person_with_ckd_any_uacr.csv")


## eGFR

Reported eGFR is curated directly from laboratory measurements. Creatinine-based eGFR is calculated using the 2021 CKD-EPI creatinine equation. Persistent reduced eGFR is defined by two values below 60 mL/min/1.73 m² at least 90 days apart.


### Reported eGFR


In [ ]:
egfr_concepts = discover_lab_concepts_by_name(
    pattern=r"glomerular filtration",
    limit=300,
    use_regex=True,
    include_source_concepts=False
)

egfr_concepts[["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]].head(40)


In [ ]:
import pandas as pd
import numpy as np

# ----------------------------
# 1) Build an allowlist for REPORTED eGFR
# ----------------------------
EGFR_EXCLUDE = (
    "panel",                    # avoid panels
    "testing",                  # procedures
    "measurement",              # procedures
    "estimation of",            # broad procedure-ish
)

def build_reported_egfr_allowlist(egfr_concepts: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only concepts that look like a numeric reported eGFR lab result.
    """
    allow = build_allowlist_from_concepts(
        concepts_df=egfr_concepts,
        keep_vocab=("LOINC", "SNOMED"),
        keep_concept_classes=("Lab Test", "Procedure"),  # we'll drop procedure-like by name filters
        must_contain=("glomerular filtration rate",),
        must_not_contain=EGFR_EXCLUDE,
        exact_allow_names=None,  # not needed; we'll filter by must_contain/must_not_contain
    )

    # Extra tightening: keep only names that clearly refer to eGFR/1.73m2 predicted
    name = allow["concept_name"].astype(str).str.lower()
    allow = allow[
        name.str.contains("glomerular filtration rate", na=False)
        & name.str.contains("1.73", na=False)  # eGFR indexed to BSA
        & name.str.contains("predicted", na=False)
    ].copy()

    return allow


# ----------------------------
# 2) Extract numeric values (reported eGFR)
# ----------------------------
def extract_reported_egfr(
    egfr_concepts: pd.DataFrame,
    min_value: float = 1.0,
    max_value: float = 200.0,
):
    egfr_allow = build_reported_egfr_allowlist(egfr_concepts)

    df_reported_egfr, egfr_allowlist = extract_numeric_measurements_from_discovered_concepts(
        concepts_df=egfr_allow,
        date_cutoffs=None,
        keep_vocab=("LOINC", "SNOMED"),
        keep_concept_classes=("Lab Test", "Procedure"),
        drop_name_contains=EGFR_EXCLUDE,
        require_units_containing=None,   # AoU often has unit nulls / inconsistent labels
        allow_unit_nulls=True,
        min_value=min_value,
        max_value=max_value,
        return_allowlist=True,
    )

    return df_reported_egfr, egfr_allowlist


# ----------------------------
# 3) Normalize reported eGFR units
# ----------------------------
def normalize_reported_egfr_units(
    df_reported_egfr_raw: pd.DataFrame,
    value_col: str = "value_as_number",
    unit_col: str = "unit_concept_name",
    out_col: str = "reported_egfr_ml_min_1p73m2",
    drop_if_unit_unknown: bool = False,
    min_value: float = 1.0,
    max_value: float = 200.0,
):
    """
    Normalize REPORTED eGFR to mL/min/1.73m^2.

    In practice, most reported eGFR is already in this unit.
    We mainly:
      - keep mL/min/1.73m2-like units as-is
      - keep unit-null if values are plausible (AoU-safe)
      - mark everything else as unknown
    """
    df = df_reported_egfr_raw.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df[unit_col] = df[unit_col].astype("string")

    unit_norm = (
        df[unit_col]
        .fillna("<NA>")
        .str.strip()
        .str.lower()
    )
    df["reported_egfr_unit_norm"] = unit_norm

    df[out_col] = np.nan
    df["reported_egfr_unit_rule"] = pd.NA

    # units we accept as "already correct"
    ok_unit = (
        unit_norm.eq("<na>") |                       # allow nulls
        unit_norm.str.contains("ml", na=False) |
        unit_norm.str.contains("milliliter", na=False) |
        unit_norm.str.contains("min", na=False) |
        unit_norm.str.contains("minute", na=False) |
        unit_norm.str.contains("1.73", na=False) |
        unit_norm.str.contains("m2", na=False)
    )

    # If unit is null OR looks like the standard eGFR unit -> keep as-is
    keep_mask = df[value_col].notna() & ok_unit
    df.loc[keep_mask, out_col] = df.loc[keep_mask, value_col]
    df.loc[keep_mask & unit_norm.eq("<na>"), "reported_egfr_unit_rule"] = "unit missing (kept if plausible)"
    df.loc[keep_mask & ~unit_norm.eq("<na>"), "reported_egfr_unit_rule"] = "ml/min/1.73m2-like (as-is)"

    # Everything else
    unknown_mask = df[out_col].isna() & df[value_col].notna()
    df.loc[unknown_mask, "reported_egfr_unit_rule"] = "unknown/unsupported unit"

    # Plausibility bounds
    bad = df[out_col].notna() & ((df[out_col] < min_value) | (df[out_col] > max_value))
    df.loc[bad, out_col] = np.nan
    df.loc[bad, "reported_egfr_unit_rule"] = f"out of range (<{min_value:g} or >{max_value:g})"

    if drop_if_unit_unknown:
        df = df.dropna(subset=[out_col])

    return df


# ----------------------------
# 4) Summarize "first low eGFR" (reported)
# ----------------------------
def summarize_first_low_reported_egfr(
    df_reported_egfr_norm: pd.DataFrame,
    threshold: float = 60.0,
    person_col: str = "person_id",
    date_col: str = "measurement_datetime",
    egfr_col: str = "reported_egfr_ml_min_1p73m2",
):
    df = df_reported_egfr_norm.copy()
    df[person_col] = df[person_col].astype("Int64")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df[egfr_col] = pd.to_numeric(df[egfr_col], errors="coerce")

    df_use = df.dropna(subset=[person_col, date_col, egfr_col]).copy()

    low = df_use[df_use[egfr_col] < threshold].copy()
    first = (
        low.sort_values([person_col, date_col])
        .groupby(person_col, as_index=False)
        .first()
        .loc[:, [person_col, egfr_col, date_col]]
        .rename(columns={
            egfr_col: "first_low_reported_egfr",
            date_col: "first_low_reported_egfr_date",
        })
    )

    print("\n" + "="*72)
    print("Reported eGFR Extraction + Normalization Summary")
    print("="*72)
    print(f"Rows input (reported eGFR):                   {len(df_reported_egfr_norm):,}")
    print(f"People with any reported eGFR (input):        {df_reported_egfr_norm[person_col].nunique():,}")
    print(f"Rows with usable normalized reported eGFR:    {len(df_use):,}")
    print(f"People with usable normalized reported eGFR:  {df_use[person_col].nunique():,}")
    print(f"People with eGFR < {threshold:g}:                         {first[person_col].nunique():,}")
    print("-"*72)

    audit = (
        df.assign(is_convertible=df[egfr_col].notna())
        .groupby("reported_egfr_unit_norm", dropna=False)
        .agg(
            n_rows=("reported_egfr_unit_norm", "size"),
            n_people=(person_col, "nunique"),
            pct_convertible=("is_convertible", "mean"),
            median_raw=("value_as_number", "median"),
            median_norm=(egfr_col, "median"),
            p95_norm=(egfr_col, lambda s: np.nanpercentile(s, 95)),
            max_norm=(egfr_col, "max"),
        )
        .reset_index()
        .sort_values("n_rows", ascending=False)
    )

    print("Unit audit (top 15 by rows):")
    print(audit.head(15).to_string(index=False))

    if len(first) > 0:
        d = pd.to_datetime(first["first_low_reported_egfr_date"], utc=True)
        qs = d.quantile([0.25, 0.5, 0.75])
        print("-"*72)
        print("First low eGFR date distribution:")
        print(f"  Earliest:  {d.min().date()}")
        print(f"  25th pct:  {qs.loc[0.25].date()}")
        print(f"  Median:    {qs.loc[0.5].date()}")
        print(f"  75th pct:  {qs.loc[0.75].date()}")
        print(f"  Latest:    {d.max().date()}")
        print("="*72)

    return first, audit


# ============================================================
# RUN (reported eGFR)
# ============================================================
df_reported_egfr_raw, egfr_allowlist = extract_reported_egfr(egfr_concepts)

df_reported_egfr_norm = normalize_reported_egfr_units(
    df_reported_egfr_raw,
    drop_if_unit_unknown=False,
    min_value=1.0,
    max_value=200.0,
)

df_first_low_reported_egfr, reported_egfr_unit_audit = summarize_first_low_reported_egfr(
    df_reported_egfr_norm,
    threshold=60.0,
)


### Computed eGFR


In [ ]:
creatinine_concepts = discover_lab_concepts_by_name(
    pattern=r"creatinine",
    limit=300,
    use_regex=True,
    include_source_concepts=False
)


In [ ]:
SERUM_CREAT_EXACT = {
    # LOINC-style
    "Creatinine [Mass/volume] in Serum or Plasma",
    "Creatinine [Moles/volume] in Serum or Plasma",
    "Creatinine [Mass/volume] in Blood",
    "Creatinine [Moles/volume] in Blood",
    "Creatinine [Mass/volume] in Arterial blood",
    "Creatinine [Moles/volume] in Serum, Plasma or Blood",
    "Creatinine [Mass/volume] in Serum, Plasma or Blood",

    # SNOMED-y (optional)
    "Creatinine measurement",
    "Creatinine measurement, serum",
    "Plasma creatinine measurement",
    "Creatinine; blood",
    "Quantitative measurement of mass concentration of creatinine in predialysis serum or plasma",
}

creat_allow_concepts_exact = (
    creatinine_concepts
    .loc[
        creatinine_concepts["concept_name"].isin(SERUM_CREAT_EXACT)
        & creatinine_concepts["vocabulary_id"].isin(["LOINC","SNOMED"])
        & creatinine_concepts["concept_class_id"].isin(["Lab Test","Procedure"]),
        ["measurement_concept_id","concept_name","vocabulary_id","concept_class_id","n_people","n_rows"]
    ]
    .drop_duplicates(subset=["measurement_concept_id"])
    .sort_values(["n_people","n_rows"], ascending=False)
    .reset_index(drop=True)
)

print("Serum creat allow concepts (EXACT):", creat_allow_concepts_exact.shape)
print(creat_allow_concepts_exact["vocabulary_id"].value_counts())
display(creat_allow_concepts_exact)


In [ ]:
df_scr_raw, scr_allowlist = extract_numeric_measurements_from_discovered_concepts(
    concepts_df=creat_allow_concepts_exact,
    date_cutoffs=None,
    keep_vocab=("LOINC","SNOMED"),
    keep_concept_classes=("Lab Test","Procedure"),
    allow_unit_nulls=True,
    min_value=0.1,
    max_value=25.0,
    return_allowlist=True,
)

print("df_scr_raw:", df_scr_raw.shape, "people:", df_scr_raw["person_id"].nunique())
print("\nTop concepts:\n", df_scr_raw["standard_concept_name"].value_counts().head(20))
print("\nTop units:\n", df_scr_raw["unit_concept_name"].value_counts().head(20))


In [ ]:
import numpy as np
import pandas as pd

def normalize_serum_creatinine(
    df_scr_raw: pd.DataFrame,
    value_col: str = "value_as_number",
    unit_col: str = "unit_concept_name",
    out_col: str = "scr_mg_dl",
    drop_if_unknown: bool = False,
):
    """
    Normalize serum creatinine to mg/dL.

    Supported:
      - mg/dL, milligram per deciliter -> as-is
      - µmol/L -> mg/dL (divide by 88.4)
      - mmol/L -> mg/dL (multiply by 113.12 / 10)
      - missing unit -> keep if value plausible

    Drops:
      - rate units (per minute, per 1.73 m2)
      - ratios, seconds, grams/day
    """

    df = df_scr_raw.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df[unit_col] = df[unit_col].astype("string")

    unit_norm = (
        df[unit_col]
        .fillna("<NA>")
        .str.strip()
        .str.lower()
    )
    df["unit_norm"] = unit_norm

    df[out_col] = np.nan
    df["scr_unit_rule"] = pd.NA

    # --- mg/dL (as-is) ---
    mgdl_mask = (
        unit_norm.str.contains("mg/dl", na=False) |
        unit_norm.str.contains("milligram per deciliter", na=False)
    )
    df.loc[mgdl_mask, out_col] = df.loc[mgdl_mask, value_col]
    df.loc[mgdl_mask, "scr_unit_rule"] = "mg/dL (as-is)"

    # --- µmol/L → mg/dL ---
    umol_mask = unit_norm.str.contains("micromole per liter", na=False)
    df.loc[umol_mask, out_col] = df.loc[umol_mask, value_col] / 88.4
    df.loc[umol_mask, "scr_unit_rule"] = "µmol/L → mg/dL (/88.4)"

    # --- mmol/L → mg/dL ---
    mmol_mask = unit_norm.str.contains("millimole per liter", na=False)
    df.loc[mmol_mask, out_col] = df.loc[mmol_mask, value_col] * 11.312
    df.loc[mmol_mask, "scr_unit_rule"] = "mmol/L → mg/dL (*11.312)"

    # --- unit missing but plausible ---
    missing_unit_mask = (
        df[out_col].isna() &
        unit_norm.isin(["<na>", "no value", "unit", "no matching concept"]) &
        df[value_col].between(0.2, 25)
    )
    df.loc[missing_unit_mask, out_col] = df.loc[missing_unit_mask, value_col]
    df.loc[missing_unit_mask, "scr_unit_rule"] = "unit missing (kept if plausible)"

    # --- everything else ---
    unknown_mask = df[out_col].isna() & df[value_col].notna()
    df.loc[unknown_mask, "scr_unit_rule"] = "unknown/unsupported unit"

    if drop_if_unknown:
        df = df.dropna(subset=[out_col])

    return df


In [ ]:
df_scr_norm = normalize_serum_creatinine(df_scr_raw, drop_if_unknown=False)

print(df_scr_norm["scr_unit_rule"].value_counts())

df_scr_norm.groupby("scr_unit_rule")["scr_mg_dl"].describe().round(2)


In [ ]:
# Save normalized serum creatinine for downstream feature construction.
save_gcs_parquet(
    df_scr_norm[
        [
            "person_id",
            "measurement_datetime",
            "scr_mg_dl",
        ]
    ].copy(),
    f"{CURATION_DIR}/longitudinal_serum_creatinine.parquet",
)


In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# 1) Demographics
# ----------------------------
demo = dataset_all_person_df[["person_id", "sex_at_birth", "date_of_birth"]].copy()
demo["sex_clean"] = demo["sex_at_birth"].map({"Female": "F", "Male": "M"}).astype("string")

demo["date_of_birth"] = pd.to_datetime(demo["date_of_birth"], errors="coerce").dt.tz_localize(None)

# ----------------------------
# 2) Merge serum creatinine (normalized mg/dL)
#   Expected columns in df_scr_norm:
#     - person_id
#     - measurement_datetime
#     - scr_mg_dl
# ----------------------------
computed_egfr_df = df_scr_norm.merge(demo, on="person_id", how="left").copy()

# measurement time -> tz-naive
computed_egfr_df["measurement_datetime"] = pd.to_datetime(
    computed_egfr_df["measurement_datetime"], errors="coerce", utc=True
)
computed_egfr_df["measurement_datetime_naive"] = (
    computed_egfr_df["measurement_datetime"].dt.tz_convert(None)
)

# ----------------------------
# 3) Age at measurement (years)
# ----------------------------
computed_egfr_df["age_years"] = (
    (computed_egfr_df["measurement_datetime_naive"] - computed_egfr_df["date_of_birth"])
    .dt.days / 365.25
)

# ----------------------------
# 4) CKD-EPI 2021 (creatinine-only, race-free) — vectorized
# ----------------------------
def egfr_ckd_epi_2021_vectorized(scr_mg_dl, age_years, sex_clean):
    scr = pd.to_numeric(scr_mg_dl, errors="coerce").astype(float)
    age = pd.to_numeric(age_years, errors="coerce").astype(float)
    sex = sex_clean.astype("string")

    out = np.full(len(scr), np.nan, dtype="float64")

    valid = scr.notna() & age.notna() & sex.notna() & (scr > 0) & (age >= 18) & (age <= 120)

    # Female
    f = valid & (sex == "F")
    if f.any():
        kappa, alpha, sex_mult = 0.7, -0.241, 1.012
        ratio = scr[f] / kappa
        out[f] = (
            142
            * (np.minimum(ratio, 1) ** alpha)
            * (np.maximum(ratio, 1) ** -1.200)
            * (0.9938 ** age[f])
            * sex_mult
        )

    # Male
    m = valid & (sex == "M")
    if m.any():
        kappa, alpha, sex_mult = 0.9, -0.302, 1.0
        ratio = scr[m] / kappa
        out[m] = (
            142
            * (np.minimum(ratio, 1) ** alpha)
            * (np.maximum(ratio, 1) ** -1.200)
            * (0.9938 ** age[m])
            * sex_mult
        )

    return out

computed_egfr_df["egfr_computed_ckd_epi_2021"] = egfr_ckd_epi_2021_vectorized(
    computed_egfr_df["scr_mg_dl"],
    computed_egfr_df["age_years"],
    computed_egfr_df["sex_clean"],
)

# ----------------------------
# 5) QC bounds
# ----------------------------
computed_egfr_df = computed_egfr_df[
    computed_egfr_df["egfr_computed_ckd_epi_2021"].between(5, 200, inclusive="both")
].copy()

print("Final rows with valid computed eGFR:", len(computed_egfr_df))
print("People with valid computed eGFR:", computed_egfr_df["person_id"].nunique())
computed_egfr_df["egfr_computed_ckd_epi_2021"].describe().round(2)


### Reusable Computed eGFR Measurements

The creatinine-derived eGFR table is saved separately because the cohort-construction notebook uses the original computed-eGFR measurement count for control surveillance eligibility.


In [ ]:
# Save the compact computed-eGFR measurement table used for
# control surveillance in Notebook 02.
computed_egfr_measurements = computed_egfr_df[
    [
        "person_id",
        "measurement_datetime",
        "egfr_computed_ckd_epi_2021",
    ]
].copy()

save_gcs_parquet(
    computed_egfr_measurements,
    f"{CURATION_DIR}/computed_egfr_measurements.parquet",
)


### Persistent Reduced eGFR


In [ ]:
import pandas as pd
import numpy as np

def prep_egfr_df(
    df: pd.DataFrame,
    egfr_col: str,
    person_col: str = "person_id",
    date_col: str = "measurement_datetime",
):
    out = df.copy()
    out[person_col] = pd.to_numeric(out[person_col], errors="coerce").astype("Int64")
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce", utc=True)
    out[egfr_col] = pd.to_numeric(out[egfr_col], errors="coerce")
    out = out.dropna(subset=[person_col, date_col, egfr_col])
    return out


In [ ]:
import pandas as pd
import numpy as np

def first_persistent_low_egfr(
    egfr_df: pd.DataFrame,
    egfr_col: str,
    threshold: float = 60.0,
    min_days_apart: int = 90,
    person_col: str = "person_id",
    date_col: str = "measurement_datetime",
    out_prefix: str = "reported",   # keep consistent with your UACR
    label_prefix: str = "egfr",
) -> pd.DataFrame:
    """
    First persistent low eGFR event:
      - eGFR < threshold
      - confirm with a second eGFR < threshold at least min_days_apart later
    Returns per person:
      - first_qualifying_low_date (first low)
      - first_persistent_low_date (confirming low date)
      - confirming_value (value at confirm date)
      - days_between
    """
    df = egfr_df.copy()

    df[person_col] = pd.to_numeric(df[person_col], errors="coerce").astype("Int64")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    df[egfr_col] = pd.to_numeric(df[egfr_col], errors="coerce")

    df = df.dropna(subset=[person_col, date_col, egfr_col]).sort_values([person_col, date_col])

    low = df.loc[df[egfr_col] < threshold, [person_col, date_col, egfr_col]].copy()
    if low.empty:
        return pd.DataFrame(columns=[
            person_col,
            f"first_qualifying_{label_prefix}_lt{int(threshold)}_{out_prefix}_date",
            f"first_persistent_{label_prefix}_lt{int(threshold)}_{out_prefix}_date",
            f"confirming_{label_prefix}_lt{int(threshold)}_{out_prefix}_value",
            "days_between",
        ])

    min_delta = pd.Timedelta(days=min_days_apart)

    def per_person(g: pd.DataFrame) -> pd.Series:
        dates = g[date_col].to_numpy()
        vals  = g[egfr_col].to_numpy()

        first_low_date = dates[0]
        for j in range(1, len(dates)):
            if (dates[j] - first_low_date) >= min_delta:
                days_between = int((dates[j] - first_low_date) / np.timedelta64(1, "D"))
                return pd.Series({
                    f"first_qualifying_{label_prefix}_lt{int(threshold)}_{out_prefix}_date": first_low_date,
                    f"first_persistent_{label_prefix}_lt{int(threshold)}_{out_prefix}_date": dates[j],
                    f"confirming_{label_prefix}_lt{int(threshold)}_{out_prefix}_value": float(vals[j]),
                    "days_between": days_between,
                })

        return pd.Series({
            f"first_qualifying_{label_prefix}_lt{int(threshold)}_{out_prefix}_date": pd.NaT,
            f"first_persistent_{label_prefix}_lt{int(threshold)}_{out_prefix}_date": pd.NaT,
            f"confirming_{label_prefix}_lt{int(threshold)}_{out_prefix}_value": np.nan,
            "days_between": np.nan,
        })

    out = low.groupby(person_col).apply(per_person).reset_index()

    # Keep only confirmed
    out = out.dropna(subset=[f"first_persistent_{label_prefix}_lt{int(threshold)}_{out_prefix}_date"]).reset_index(drop=True)

    # Stable column order
    out = out[[
        person_col,
        f"first_qualifying_{label_prefix}_lt{int(threshold)}_{out_prefix}_date",
        f"first_persistent_{label_prefix}_lt{int(threshold)}_{out_prefix}_date",
        f"confirming_{label_prefix}_lt{int(threshold)}_{out_prefix}_value",
        "days_between",
    ]]
    return out


In [ ]:
reported_egfr_clean = prep_egfr_df(
    df_reported_egfr_norm,
    egfr_col="value_as_number"
)

ckd_reported_egfr = first_persistent_low_egfr(
    egfr_df=reported_egfr_clean,
    egfr_col="value_as_number",
    threshold=60.0,
    min_days_apart=90,
    label_prefix="reported_egfr",
)

print(
    "People with CKD by reported eGFR:",
    ckd_reported_egfr["person_id"].nunique()
)


In [ ]:
computed_egfr_clean = prep_egfr_df(
    computed_egfr_df,
    egfr_col="egfr_computed_ckd_epi_2021"
)

ckd_computed_egfr = first_persistent_low_egfr(
    egfr_df=computed_egfr_clean,
    egfr_col="egfr_computed_ckd_epi_2021",
    threshold=60.0,
    min_days_apart=90,
    label_prefix="computed_egfr",
)

print("People with CKD by computed eGFR:",
      ckd_computed_egfr["person_id"].nunique())


In [ ]:
rep = ckd_reported_egfr.rename(columns={
    "first_persistent_reported_egfr_lt60_reported_date": "ckd_egfr_onset_date",
    "confirming_reported_egfr_lt60_reported_value": "ckd_egfr_value",
}).assign(ckd_egfr_source="reported")

cmp = ckd_computed_egfr.rename(columns={
    "first_persistent_computed_egfr_lt60_reported_date": "ckd_egfr_onset_date",
    "confirming_computed_egfr_lt60_reported_value": "ckd_egfr_value",
}).assign(ckd_egfr_source="computed")

ckd_egfr_any = pd.concat(
    [
        rep[["person_id", "ckd_egfr_onset_date", "ckd_egfr_value", "ckd_egfr_source"]],
        cmp[["person_id", "ckd_egfr_onset_date", "ckd_egfr_value", "ckd_egfr_source"]],
    ],
    ignore_index=True,
)

# Make dates consistent (pick one convention; I suggest naive)
ckd_egfr_any["ckd_egfr_onset_date"] = (
    pd.to_datetime(ckd_egfr_any["ckd_egfr_onset_date"], errors="coerce", utc=True)
      .dt.tz_convert(None)
)

# Drop missing dates before collapsing
ckd_egfr_any = ckd_egfr_any.dropna(subset=["ckd_egfr_onset_date"])

# Deterministic earliest onset; tie-break: prefer computed if same day
source_rank = {"computed": 0, "reported": 1}
ckd_egfr_any["source_rank"] = ckd_egfr_any["ckd_egfr_source"].map(source_rank).astype(int)

ckd_egfr_any = (
    ckd_egfr_any
    .sort_values(["person_id", "ckd_egfr_onset_date", "source_rank"])
    .groupby("person_id", as_index=False)
    .first()
    .drop(columns=["source_rank"])
)

print("People with CKD by eGFR (reported OR computed):",
      ckd_egfr_any["person_id"].nunique())


In [ ]:
# Save
save_phenotypes(ckd_egfr_any, bucket, "person_with_ckd_any_egfr.csv")


## Kidney Replacement Therapy

Dialysis and kidney transplantation are retained as dated kidney disease events.


### Kidney Transplantation


In [ ]:
dialysis_icd10 = [
    "Z99.2",       # Dependence on renal dialysis
    "Z49.01",      # Encounter for fitting/adjustment of extracorporeal dialysis catheter
    "Z49.02",      # Encounter for fitting/adjustment of peritoneal dialysis catheter
    "Z49.31",      # Encounter for adequacy testing for hemodialysis
    "Z49.32",      # Encounter for adequacy testing for peritoneal dialysis
    "Z49.01", "Z49.02", "Z49.31", "Z49.32",
    "Z49.81",      # Encounter for vascular access for dialysis
    "Z49.89"       # Other specified dialysis encounter
]
dialysis_icd9 = [
    "V45.1",       # Renal dialysis status
    "V56",         # Encounter for dialysis & care involving dialysis
    "V56.0", "V56.1", "V56.2", "V56.3", "V56.8"
]
dialysis_cpt = [
    # Hemodialysis
    "90935", "90937", "90940",
    # Monthly dialysis care
    "90960", "90961", "90962", "90965", "90966",
    # Other dialysis services
    "90999"
]


transplant_icd10 = [
    "Z94.0",    # Kidney transplant status
    "T86.1",    # Kidney transplant rejection
    "T86.10", "T86.11", "T86.12", "T86.19",  # Specific rejection/complications
    "Y83.0"     # Surgical operation with transplant of whole organ
]
transplant_icd9 = [
    "V42.0",    # Kidney transplant status
    "996.81"    # Complications of transplanted kidney
]
transplant_cpt = [
    "50360",    # Kidney transplant, with recipient nephrectomy
    "50365",    # Kidney transplant, without recipient nephrectomy
    "50370",    # Reimplantation of transplanted kidney
    "50380"     # Removal of transplanted kidney
]

# --- Dialysis & transplant codes combined ---
esrd_codes = (
    dialysis_icd10 + dialysis_icd9 + dialysis_cpt +
    transplant_icd10 + transplant_icd9 + transplant_cpt
)


In [ ]:
ckd_replac = subset_code_exact_match_for_terms(dataset_all_condition_sql, esrd_codes, 'person_id', 'condition_start_datetime')


In [ ]:
ckd_replac_earliest = subset_unique_earliest_date(ckd_replac, 'person_id', 'condition_start_datetime')


In [ ]:
# Save
save_phenotypes(ckd_replac_earliest, bucket, "ckd_dignosis_transplant_codes.csv")


### Dialysis Measurements


In [ ]:
from google.cloud import bigquery
import pandas as pd

def subset_labs_multi_partial_str_at_any_date(
    sql_query,
    search_terms,
    id_column,
    date_column,
    name_column="standard_concept_name",
    exclude_terms=None,
):
    """
    Subset rows where `name_column` contains ANY of `search_terms` (case-insensitive).
    Optionally exclude rows where `name_column` contains ANY of `exclude_terms`.
    Keeps a ROW_NUMBER() per id ordered by date desc (like your original).
    """
    client = bigquery.Client()

    # Guard: listify single strings
    if isinstance(search_terms, str):
        search_terms = [search_terms]
    exclude_terms = exclude_terms or []
    if isinstance(exclude_terms, str):
        exclude_terms = [exclude_terms]

    # Simple SQL-safe lowercasing + quote escaping
    esc = lambda s: s.replace("'", "''").lower()

    include_clause = " OR ".join(
        [f"LOWER({name_column}) LIKE '%{esc(t)}%'" for t in search_terms]
    )

    exclude_clause = ""
    if exclude_terms:
        exclude_clause = " AND " + " AND ".join(
            [f"LOWER({name_column}) NOT LIKE '%{esc(t)}%'" for t in exclude_terms]
        )

    subset_query = f"""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY {id_column} ORDER BY {date_column} DESC) AS row_num
        FROM (
            {sql_query}
        )
        WHERE ({include_clause})
        {exclude_clause}
    )
    """

    df = client.query(subset_query).to_dataframe()

    # Date cleanup to match your original
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")
    df["measurement_date"] = df[date_column].dt.date

    # Remove known sentinel junk
    if "value_as_number" in df.columns:
        df = df[df["value_as_number"] != 10000000.0]

    return df


In [ ]:
dataset_32981317_measurement_sql = f"""
    SELECT
        m.person_id,
        m.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        m.measurement_datetime,
        m.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        m.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        m.value_as_number,
        m.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        m.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        m.range_low,
        m.range_high,
        m.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        m.measurement_source_value,
        m.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        m.unit_source_value,
        m.value_source_value 
    FROM
        `{CDR_DATASET}.measurement` m
    LEFT JOIN
        `{CDR_DATASET}.concept` m_standard_concept
        ON m.measurement_concept_id = m_standard_concept.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_type
        ON m.measurement_type_concept_id = m_type.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_operator
        ON m.operator_concept_id = m_operator.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_value
        ON m.value_as_concept_id = m_value.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_unit
        ON m.unit_concept_id = m_unit.concept_id
    LEFT JOIN
        `{CDR_DATASET}.visit_occurrence` v
        ON m.visit_occurrence_id = v.visit_occurrence_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_visit
        ON v.visit_concept_id = m_visit.concept_id
    LEFT JOIN
        `{CDR_DATASET}.concept` m_source_concept
        ON m.measurement_source_concept_id = m_source_concept.concept_id
    WHERE
        m.value_as_number IS NOT NULL
"""


In [ ]:
dialysis_lab_terms = ["dialysis"]


dialysis_lab = subset_labs_multi_partial_str_at_any_date(
    sql_query=dataset_32981317_measurement_sql,   # your base SQL
    search_terms=dialysis_lab_terms,
    exclude_terms=None,
    id_column="person_id",
    date_column="measurement_datetime",
    name_column="standard_concept_name",
)


In [ ]:
# Get the earliest (first) lab per person
first_dialysis_lab = (
    dialysis_lab.sort_values(["person_id", "measurement_datetime"])
    .groupby("person_id")
    .first()
    .reset_index()
    .loc[:, ["person_id", "value_as_number", "standard_concept_name", "measurement_datetime"]]
)


In [ ]:
# Save
save_phenotypes(first_dialysis_lab, bucket, "ckd_dignosis_dialysis_labs.csv")


### Dialysis Diagnosis and Procedure Codes


In [ ]:
from google.cloud import bigquery
import pandas as pd

def subset_condition_2terms_source(sql_query, search_term1, search_term2):
    # Set up the BigQuery client
    client = bigquery.Client()

    # Convert the search terms to lowercase
    lower_search_term1 = search_term1.lower()
    lower_search_term2 = search_term2.lower()

    # Define the query to subset the rows based on the desired condition
    subset_query = f"""
    SELECT *
    FROM (
        {sql_query}
    )
    WHERE LOWER(source_concept_name) LIKE '%' || LOWER('{lower_search_term1}') || '%'
       OR LOWER(source_concept_name) LIKE '%' || LOWER('{lower_search_term2}') || '%'
    """


    # Execute the query and retrieve the results
    subset_query_job = client.query(subset_query)
    subset_results = subset_query_job.result()

    # Create an empty dictionary to store the columns
    columns = {}

    # Iterate over the results and extract the columns
    for i, row in enumerate(subset_results):
        for col in row.keys():
            if col not in columns:
                columns[col] = []
            columns[col].append(row[col])

    # Create a DataFrame from the extracted columns
    subset_df = pd.DataFrame(columns)

    return subset_df


In [ ]:
dialysis_code = subset_condition_2terms_source(dataset_all_condition_sql,
                                                 'dialysis', 'hemodialysis')


In [ ]:
# Relevant keywords for renal dialysis
dialysis_renal_terms = ["renal dialysis", "hemodialysis", "peritoneal dialysis", "ckd", "esrd", "end stage"]

def classify_dialysis_term(name):
    n = name.lower()
    if "iridodialysis" in n or "retinal" in n:
        return "exclude_eye"
    if any(t in n for t in dialysis_renal_terms):
        return "keep_renal"
    if "dialysis catheter" in n:
        return "keep_device"
    return "exclude_other"

dialysis_code["dialysis_class"] = dialysis_code["source_concept_name"].apply(classify_dialysis_term)

# Keep only real dialysis or device-related
dialysis_code_clean = dialysis_code.loc[dialysis_code["dialysis_class"].isin(["keep_renal","keep_device"])]


In [ ]:
dialysis_code_clean_earliest = subset_unique_earliest_date(dialysis_code_clean, 'person_id', 'condition_start_datetime')


In [ ]:
# Save
save_phenotypes(dialysis_code_clean_earliest, bucket, "ckd_dignosis_dialysis_codes.csv")


# 5. Reusable Longitudinal Renal Data

Reported and computed renal measurements are combined into standardized longitudinal tables. These measurement-level datasets are reused for renal surveillance and pre-landmark feature construction without rerunning the underlying extraction.


## Combined Longitudinal UACR


In [ ]:
def build_combined_uacr_longitudinal(
    df_reported,
    df_computed
):
    # ---- reported ----
    rep = df_reported[[
        "person_id",
        "measurement_datetime",
        "uacr_mg_g"
    ]].copy()

    rep["person_id"] = rep["person_id"].astype("Int64")
    rep["measurement_datetime"] = pd.to_datetime(
        rep["measurement_datetime"], errors="coerce", utc=True
    )
    rep["uacr_mg_g"] = pd.to_numeric(rep["uacr_mg_g"], errors="coerce")
    rep["uacr_source"] = "reported"

    rep = rep.dropna(subset=["person_id", "measurement_datetime", "uacr_mg_g"])

    # ---- computed ----
    comp = df_computed[[
        "person_id",
        "measurement_datetime_alb",
        "uacr_computed_mg_g"
    ]].copy()

    comp = comp.rename(columns={
        "measurement_datetime_alb": "measurement_datetime",
        "uacr_computed_mg_g": "uacr_mg_g"
    })

    comp["person_id"] = comp["person_id"].astype("Int64")
    comp["measurement_datetime"] = pd.to_datetime(
        comp["measurement_datetime"], errors="coerce", utc=True
    )
    comp["uacr_mg_g"] = pd.to_numeric(comp["uacr_mg_g"], errors="coerce")
    comp["uacr_source"] = "computed"

    comp = comp.dropna(subset=["person_id", "measurement_datetime", "uacr_mg_g"])

    # ---- combine ----
    uacr_long = pd.concat([rep, comp], ignore_index=True)

    # remove impossible values
    uacr_long = uacr_long[uacr_long["uacr_mg_g"] >= 0]

    # optional: drop extreme outliers if needed later
    # uacr_long = uacr_long[uacr_long["uacr_mg_g"] < 5000]

    return uacr_long


In [ ]:
df_uacr_long = build_combined_uacr_longitudinal(
    df_reported=df_reported_uacr_norm,
    df_computed=df_uacr_computed,
)

df_uacr_long = (
    df_uacr_long
    .sort_values(
        [
            "person_id",
            "measurement_datetime",
            "uacr_mg_g",
        ]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "measurement_datetime",
            "uacr_mg_g",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    f"Longitudinal UACR observations: {len(df_uacr_long):,}"
)
print(
    f"Participants with UACR data: "
    f"{df_uacr_long['person_id'].nunique():,}"
)

save_gcs_parquet(
    df_uacr_long,
    f"{CURATION_DIR}/longitudinal_uacr_measurements.parquet",
)


## Combined Longitudinal eGFR


In [ ]:
def build_combined_egfr_longitudinal(
    df_reported_egfr_norm: pd.DataFrame,
    computed_egfr_df: pd.DataFrame,
    reported_value_col: str = "reported_egfr_ml_min_1p73m2",
    computed_value_col: str = "egfr_computed_ckd_epi_2021",
    dt_col: str = "measurement_datetime",
):
    # ---- reported ----
    rep = df_reported_egfr_norm[["person_id", dt_col, reported_value_col]].copy()
    rep = rep.rename(columns={reported_value_col: "egfr_ml_min_1p73m2"})
    rep["person_id"] = pd.to_numeric(rep["person_id"], errors="coerce").astype("Int64")
    rep[dt_col] = pd.to_datetime(rep[dt_col], errors="coerce", utc=True)
    rep["egfr_ml_min_1p73m2"] = pd.to_numeric(rep["egfr_ml_min_1p73m2"], errors="coerce")
    rep["egfr_source"] = "reported"
    rep = rep.dropna(subset=["person_id", dt_col, "egfr_ml_min_1p73m2"])

    # ---- computed ----
    comp = computed_egfr_df[["person_id", dt_col, computed_value_col]].copy()
    comp = comp.rename(columns={computed_value_col: "egfr_ml_min_1p73m2"})
    comp["person_id"] = pd.to_numeric(comp["person_id"], errors="coerce").astype("Int64")
    comp[dt_col] = pd.to_datetime(comp[dt_col], errors="coerce", utc=True)
    comp["egfr_ml_min_1p73m2"] = pd.to_numeric(comp["egfr_ml_min_1p73m2"], errors="coerce")
    comp["egfr_source"] = "computed"
    comp = comp.dropna(subset=["person_id", dt_col, "egfr_ml_min_1p73m2"])

    # ---- combine ----
    egfr_long = pd.concat([rep, comp], ignore_index=True)

    # basic sanity filters
    egfr_long = egfr_long[egfr_long["egfr_ml_min_1p73m2"] >= 0]

    # optional: de-dup exact duplicates
    egfr_long = (
        egfr_long
        .sort_values(["person_id", dt_col, "egfr_ml_min_1p73m2"])
        .drop_duplicates(subset=["person_id", dt_col, "egfr_ml_min_1p73m2"], keep="first")
    )

    return egfr_long


In [ ]:
df_egfr_long = build_combined_egfr_longitudinal(
    df_reported_egfr_norm=df_reported_egfr_norm,
    computed_egfr_df=computed_egfr_df,
    reported_value_col="reported_egfr_ml_min_1p73m2",
    computed_value_col="egfr_computed_ckd_epi_2021",
    dt_col="measurement_datetime",
)

df_egfr_long = (
    df_egfr_long
    .sort_values(
        [
            "person_id",
            "measurement_datetime",
            "egfr_ml_min_1p73m2",
        ]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "measurement_datetime",
            "egfr_ml_min_1p73m2",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    f"Longitudinal eGFR observations: {len(df_egfr_long):,}"
)
print(
    f"Participants with eGFR data: "
    f"{df_egfr_long['person_id'].nunique():,}"
)

save_gcs_parquet(
    df_egfr_long,
    f"{CURATION_DIR}/longitudinal_egfr_measurements.parquet",
)


# 6. Curated Output Quality Control

Participant-level phenotype tables are checked for unique participant identifiers where expected, valid event dates, and the required schemas used by cohort construction.


In [ ]:
# Participant-level tables expected to contain one row per participant.
participant_level_tables = {
    "ehr_dates": ehr_dates,
    "final_t2d_dx": final_t2d_dx,
    "ckd_icd_earliest": ckd_icd_earliest,
    "dkd_icd_earliest": dkd_icd_earliest,
    "either_uacr": either_uacr,
    "ckd_egfr_any": ckd_egfr_any,
    "first_dialysis_lab": first_dialysis_lab,
    "dialysis_code_clean_earliest": dialysis_code_clean_earliest,
    "ckd_replac_earliest": ckd_replac_earliest,
}

for name, df in participant_level_tables.items():
    assert "person_id" in df.columns, f"{name}: missing person_id"
    assert not df["person_id"].duplicated().any(), (
        f"{name}: duplicate participant rows detected"
    )
    print(
        f"{name:<34} "
        f"N={df['person_id'].nunique():,}"
    )

required_egfr_cols = {
    "person_id",
    "measurement_datetime",
    "egfr_ml_min_1p73m2",
    "egfr_source",
}

required_uacr_cols = {
    "person_id",
    "measurement_datetime",
    "uacr_mg_g",
    "uacr_source",
}

assert required_egfr_cols.issubset(df_egfr_long.columns)
assert required_uacr_cols.issubset(df_uacr_long.columns)

assert (
    df_egfr_long["egfr_ml_min_1p73m2"] >= 0
).all()

assert (
    df_uacr_long["uacr_mg_g"] >= 0
).all()

print("\nAll curated-data QC checks passed.")


## Output Manifest

The following files form the stable interface between data curation and cohort construction.


In [ ]:
OUTPUT_PATHS = {
    "demographics": (
        f"{CURATION_DIR}/participant_demographics.parquet"
    ),
    "ehr_dates": (
        f"{CURATION_DIR}/derived_ehr_dates.csv"
    ),
    "t2d_first_evidence": (
        f"{CURATION_DIR}/final_t2d_first_evidence.csv"
    ),
    "ckd_icd": (
        f"{CURATION_DIR}/ckd_dignosis_icd_codes.csv"
    ),
    "dkd_icd": (
        f"{CURATION_DIR}/dkd_dignosis_icd_codes.csv"
    ),
    "persistent_uacr": (
        f"{CURATION_DIR}/person_with_ckd_any_uacr.csv"
    ),
    "persistent_egfr": (
        f"{CURATION_DIR}/person_with_ckd_any_egfr.csv"
    ),
    "dialysis_codes": (
        f"{CURATION_DIR}/ckd_dignosis_dialysis_codes.csv"
    ),
    "dialysis_labs": (
        f"{CURATION_DIR}/ckd_dignosis_dialysis_labs.csv"
    ),
    "transplant": (
        f"{CURATION_DIR}/ckd_dignosis_transplant_codes.csv"
    ),
    "longitudinal_uacr": (
        f"{CURATION_DIR}/longitudinal_uacr_measurements.parquet"
    ),
    "longitudinal_egfr": (
        f"{CURATION_DIR}/longitudinal_egfr_measurements.parquet"
    ),
    "computed_egfr": (
        f"{CURATION_DIR}/computed_egfr_measurements.parquet"
    ),
    "serum_creatinine": (
        f"{CURATION_DIR}/longitudinal_serum_creatinine.parquet"
    ),
    "spot_urine_albumin": (
        f"{CURATION_DIR}/longitudinal_spot_urine_albumin.parquet"
    ),
    "a1c_labs": (
        f"{CURATION_DIR}/aou_a1c_labs.csv"
    ),
    "fbg_labs": (
        f"{CURATION_DIR}/aou_fbg_labs.csv"
    ),
    "rbg_labs": (
        f"{CURATION_DIR}/all_random_blood_glucose.parquet"
    ),
    "basic_surveys": (
        f"{CURATION_DIR}/aou_basic_surveys.csv"
    ),
    "social_health_surveys": (
        f"{CURATION_DIR}/aou_social_health_surveys.csv"
    ),
    "lifestyle_surveys": (
        f"{CURATION_DIR}/aou_lifestyle_surveys.csv"
    ),
    "personal_family_history_surveys": (
        f"{CURATION_DIR}/aou_personal_family_history_surveys.csv"
    ),
}

output_manifest = pd.DataFrame(
    [
        {
            "dataset": name,
            "path": path,
            "exists": fs.exists(path),
        }
        for name, path in OUTPUT_PATHS.items()
    ]
)

output_manifest


In [ ]:
# Save the manifest so downstream notebooks can audit their inputs.
save_gcs_csv(
    output_manifest,
    f"{CURATION_DIR}/data_curation_output_manifest.csv",
)
